# 📏 22 · Sampling: the law of large numbers, the CLT and confidence intervals
### Data as i.i.d. random variables, the sample mean and its standard error, the law of large numbers, the central limit theorem, sampling distributions, confidence intervals, sample-size planning and the bootstrap

**BiologyToMath** · Part F — Statistics: from data to estimates · Module 22 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/22_sampling_lln_clt_confidence.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`), pandas, Matplotlib |
| **Prerequisites** | [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (Bernoulli, binomial, Poisson, normal and exponential distributions), [Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) (expectation, variance and their rules — used in every derivation here); limits from [Module 14 · 📈 Calculus: derivatives, integrals and rates of change](14_calculus_derivatives_integrals.ipynb) help |
| **Estimated time** | 160 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Statistics in a nutshell (collecting, describing and analysing data; "Is it fair?" and "Is it normally distributed?"); what a statistic is (point and interval statistics; a statistic is a random variable); i.i.d. data and the sample mean $\bar X_n$ as a random variable (the 100-coin example); $E[\bar X_n] = \mu$, $\operatorname{Var}(\bar X_n) = \sigma^2/n$; the law of large numbers and its precise form $P(|\bar X_n - \mu| < \varepsilon) \to 1$; the central limit theorem for the mean and the sum, standardisation, and why the CLT is used; sampling distributions and standard errors; confidence intervals for a mean (z and t), their interpretation and sample-size planning; the bootstrap |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All data in this notebook are **SIMULATED** with known truth: mRNA counts per cell (a gamma–Poisson model of transcriptional bursting), coin flips, genotypes, a polygenic trait, replicate measurements and the cholesterol trial of the toolkit. The record of 100 coin flips in Section 1 is typed in. Effect sizes, standard deviations and margins used for sample-size calculations are illustrative.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a *statistic* is, why it is itself a random variable, and how statistics fits into the three phases of
   a study (collect, describe, analyse).
2. **Derive** $E[\bar X_n] = \mu$ and $\operatorname{Var}(\bar X_n) = \sigma^2/n$ for i.i.d. data and **compute** the
   standard error $\sigma/\sqrt n$.
3. **State** the law of large numbers precisely ($P(|\bar X_n - \mu| < \varepsilon) \to 1$), **prove** it with Chebyshev's
   inequality, and **explain** why "$P(\bar X_n = \mu) \to 1$" is wrong.
4. **State** the central limit theorem with the correct parameters ($\bar X_n \approx \mathcal N(\mu, \sigma^2/n)$),
   **standardise** a mean or a sum, and **verify** the theorem by simulation from skewed distributions.
5. **Use** the CLT to approximate probabilities that would be hard to compute exactly, and **judge** whether a sample is
   large enough.
6. **Construct** and **interpret** z and t confidence intervals for a mean — including what a 95 % interval does *not*
   mean — and **check** their coverage by simulation.
7. **Plan** a sample size for a desired margin of error and **allocate** biological vs technical replicates.
8. **Compute** a bootstrap confidence interval when no formula is available.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, simulate mRNA counts per cell | 5 min |
| 1 · Statistics in a nutshell | three phases; "Is it fair?" (100 typed-in flips); "Is it normally distributed?"; what a statistic is | 15 min |
| 2 · The sample mean is a random variable | i.i.d. data; derive $E[\bar X_n]$, $\operatorname{Var}(\bar X_n)$; the standard error | 20 min |
| 3 · The law of large numbers | running means; the precise statement; Chebyshev's proof; the Cauchy warning | 25 min |
| 4 · The central limit theorem | standardisation; histograms for skewed data at n = 1, 2, 5, 30; polygenic traits | 25 min |
| 5 · Why use the CLT | exact vs approximate probabilities; is the sample size adequate? | 15 min |
| 6 · Sampling distributions | mean, median, SD; the standard error of an allele frequency | 15 min |
| 7 · Confidence intervals | z intervals, coverage of 100 intervals, what 95 % means, t intervals, the cholesterol trial | 25 min |
| 8 · Planning a study | sample size from a margin of error; technical vs biological replicates | 10 min |
| 9 · The bootstrap | resampling the data to get an interval for a median | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A cell biologist counts the molecules of one messenger RNA in individual cells with single-molecule fluorescence in situ
hybridisation (smFISH): each dot under the microscope is one mRNA. Because genes are transcribed in **bursts** — a
promoter switches on, fires off a batch of transcripts, then falls silent — the counts are very uneven: many cells contain
a handful of molecules, a few contain dozens (Raj *et al.*, 2006). She counts 30 cells and finds an average of about 6
molecules per cell. Three questions follow immediately:

* **How close** is the average of 30 cells to the true mean of the whole tissue — and would 300 cells really be better?
* The counts are badly skewed, yet every statistics textbook draws bell curves. **Why is the bell curve the right picture
  for the average**, even when it is wrong for the individual cells?
* What error bar should she report — and **how many cells (or mice) should the next experiment use**?

The answers are three theorems and one technique: the **law of large numbers** (averages settle down to the truth), the
**central limit theorem** (averages are approximately normal, with a spread that shrinks like $1/\sqrt n$), **confidence
intervals** (an error bar with a guaranteed long-run success rate) and the **bootstrap** (an error bar computed by
resampling the data when no formula exists).

| Biological question | Mathematical tool | Where in this module |
|---|---|---|
| Is a coin (or a 50:50 sex ratio, or a Mendelian 1:1 cross) fair? | the sample proportion as a statistic | 1, 5 |
| How precise is the mean of $n$ cells? | $\operatorname{Var}(\bar X_n) = \sigma^2/n$, standard error | 2, 6 |
| Does averaging more replicates always help? | law of large numbers (and when it fails) | 3 |
| Why are height, LDL cholesterol and other polygenic traits bell-shaped? | central limit theorem | 4 |
| What is the probability that 50 cells contain more than 250 transcripts? | normal approximation to a sum | 5 |
| What error bar for the mean LDL change in a drug trial? | z and t confidence intervals | 7 |
| How many mice, how many technical replicates? | sample-size formula, variance components | 8 |
| An error bar for a median? | the bootstrap | 9 |

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every number in this notebook is generated by
> code. The mRNA counts come from a gamma–Poisson ("negative binomial") model of transcriptional bursting with a true
> mean of 6 molecules per cell and a true variance of 30. Because we built the "tissue" ourselves, we know the truth and
> can check, thousands of times over, whether averages, intervals and bootstrap error bars behave as the theory promises —
> something no real experiment can do.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from math import comb, sqrt
from scipy import stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). They simulate the paired cholesterol drug trial (LDL before and after treatment, with a hidden true effect of −18 mg/dL) and Hardy–Weinberg genotypes at one SNP, both with fixed default seeds. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd


def simulate_genotypes(n: int, p: float, inbreeding: float = 0.0, seed: int = 1866) -> np.ndarray:
    """SIMULATED genotypes of ``n`` diploid individuals at one SNP, coded 0/1/2 = number of copies of
    the allele with frequency ``p``.  With ``inbreeding`` F = 0 the genotype frequencies follow
    Hardy–Weinberg proportions ((1−p)², 2p(1−p), p²); F > 0 moves individuals from heterozygotes to
    homozygotes: P(1) = 2p(1−p)(1−F)."""
    rng = np.random.default_rng(seed)
    q = 1.0 - p
    probs = np.array([q * q + inbreeding * p * q, 2 * p * q * (1 - inbreeding), p * p + inbreeding * p * q])
    return rng.choice(3, size=n, p=probs / probs.sum())


def simulate_cholesterol_trial(n: int = 12, effect: float = -18.0, seed: int = 2026) -> pd.DataFrame:
    """SIMULATED paired drug trial: LDL cholesterol (mg/dL) of ``n`` patients before and after
    treatment.  Patients differ a lot from each other (between-patient SD 30 mg/dL around 160), the
    measurement/day-to-day noise is smaller (SD 10), and the drug changes LDL by ``effect`` on
    average (the hidden truth).  Columns: patient, before, after."""
    rng = np.random.default_rng(seed)
    level = rng.normal(160.0, 30.0, n)
    before = level + rng.normal(0.0, 10.0, n)
    after = level + effect + rng.normal(0.0, 10.0, n)
    return pd.DataFrame({"patient": [f"P{i + 1:02d}" for i in range(n)],
                         "before": before.round(1), "after": after.round(1)})

print("🧰 toolkit loaded:", ", ".join(sorted(['simulate_cholesterol_trial', 'simulate_genotypes'])))

In [ ]:
# ── 0.1 A tissue of cells with bursty transcription (SIMULATED) ──
TRUE_MEAN = 6.0                  # true mean number of mRNA molecules per cell (the parameter μ we want)
BURST_SHAPE = 1.5                # small shape = very bursty = very uneven counts
TRUE_VAR = TRUE_MEAN + TRUE_MEAN ** 2 / BURST_SHAPE      # variance of a gamma–Poisson count
TRUE_SD = sqrt(TRUE_VAR)


def simulate_mrna_per_cell(n_cells, rng, mean=TRUE_MEAN, burst_shape=BURST_SHAPE):
    """SIMULATED smFISH counts: each cell gets a burst-driven rate λ ~ Gamma(shape, mean/shape), then a Poisson(λ) count.
    The counts are negative binomial with mean `mean` and variance mean + mean²/shape."""
    rates = rng.gamma(burst_shape, mean / burst_shape, size=n_cells)
    return rng.poisson(rates)


cells = simulate_mrna_per_cell(30, rng)
print("🧪 truth: μ =", TRUE_MEAN, "molecules per cell, σ² =", TRUE_VAR, f"(σ = {TRUE_SD:.3f})")
print("one experiment, 30 cells:", cells.tolist())
print(f"sample mean = {cells.mean():.3f}, sample SD = {cells.std(ddof=1):.3f}, "
      f"min = {cells.min()}, max = {cells.max()}")

🔍 **What just happened?** One simulated experiment: 30 cells with between 0 and 20 molecules each. The sample
mean, 5.733, is close to — but not equal to — the true mean 6; the sample standard deviation 4.448 is close to the true
$\sigma = 5.477$. Notice how uneven the counts are: the standard deviation is almost as large as the mean, so a single
cell tells you little. The rest of the notebook asks how much 30 cells together tell you.

---
## 1 · Statistics in a nutshell, and what a statistic is

> 📍 **Where we are.** Modules 16–21 built probability *forwards*: given a distribution, what data do we expect? Statistics
> runs *backwards*: given data, what can we say about the distribution that produced them? This short section sets up
> the vocabulary that Modules 22–27 use.

**Statistics deals with data.** Its goal is to **make inferences based on data**. The process has three overlapping phases:

1. **Collecting data** — run experiments and record the results. This phase is *preceded by forming hypotheses* about the
   phenomenon of interest: you decide what you want to learn before you collect (and before you look).
2. **Describing data** — summarise the results: tables, plots, averages, spreads.
3. **Analysing data** — infer from the results the *strength of the evidence* with respect to the hypotheses.

Two questions that sound simple make this concrete.

**Example 1 · "Is it fair?"** A coin is flipped 100 times. We code each flip as a random variable $X_i$ with value 1 for
heads (H) and 0 for tails (T), and summarise the 100 values by their average $\bar X_{100}$ — the fraction of heads. A
fair coin should give about one half. The same question appears in biology whenever we expect a 1:1 ratio: a 50:50 sex
ratio among offspring, a Mendelian test cross (Aa × aa gives Aa : aa = 1 : 1), or the two alleles of a heterozygote
being read equally often in RNA-seq.

In [ ]:
# ── 1.1 "Is it fair?" — a typed-in record of 100 coin flips ──
flip_record = ["TTTHHTHHHT", "HTHTHTHTHT", "HTTTHTTTTH", "HTTHHTHHTH", "TTHHHHTHTH",
               "TTTHTHHHHT", "TTTHHHTTTH", "HHHHHHHTTT", "HTHHTTTHHT", "HTHHHTTTHH"]
flips = np.array([1 if c == "H" else 0 for row in flip_record for c in row])     # 1 = heads, 0 = tails
for row in flip_record:
    print(" ".join(row))
n_heads, n = flips.sum(), len(flips)
print(f"\nn = {n} flips: {n_heads} heads, {n - n_heads} tails")
print(f"X̄_100 = (X_1 + … + X_100) / 100 = {n_heads}/{n} = {flips.mean():.2f}   (fraction of heads)")
print(f"fraction of tails = {n - n_heads}/{n} = {1 - flips.mean():.2f}")

🔍 **What just happened?** Counting the typed-in record gives 52 heads and 48 tails, so with heads coded as 1 the sample
mean is $\bar X_{100} = 52/100 = 0.52$ (the fraction of *tails* is 0.48). Is 0.52 "about one half"? We cannot answer yet:
we need to know how much $\bar X_{100}$ wobbles from experiment to experiment when the coin *is* fair. Sections 2–5 work
this out (the answer: by about ±0.05, so 0.52 is entirely unremarkable), and Module 25 turns it into a formal test.

> ⚠️ **Common confusion.** Always say which outcome is coded as 1. "48 out of 100" and "52 out of 100" describe the same
> record — one counts tails, the other heads. A proportion without its coding is ambiguous.

**Example 2 · "Is it normally distributed?"** Many methods (t-tests, linear regression) assume normally distributed data.
Below are 100 SIMULATED measurements — think of them as the log fold change of a gene in 100 tumours relative to normal
tissue. We describe them with a histogram, the **sample average** $\bar X_n$ and the **sample standard deviation**
$s_n$, and compare with the normal curve that has the same mean and SD.

In [ ]:
# ── 1.2 "Is it normally distributed?" — 100 SIMULATED log fold changes ──
rng_lfc = np.random.default_rng(SEED + 14)
lfc = stats.skewnorm.rvs(a=5, loc=-1.6, scale=2.55, size=100, random_state=rng_lfc)   # a right-skewed sample
x_bar, s_n = lfc.mean(), lfc.std(ddof=1)
print(f"sample average X̄_n = {x_bar:.2f}, sample standard deviation s_n = {s_n:.2f}, "
      f"skewness = {stats.skew(lfc):.2f} (0 for a normal distribution)")
inside_1sd = np.mean(np.abs(lfc - x_bar) < s_n)
print(f"fraction within 1 SD of the mean: {inside_1sd:.2f} (normal: 0.68); min = {lfc.min():.2f}, max = {lfc.max():.2f}")

grid = np.linspace(-4.5, 6, 300)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(lfc, bins=np.arange(-4, 6.5, 1.0), density=True, color="tab:cyan", edgecolor="k", alpha=0.7, label="data (n = 100)")
ax.plot(grid, stats.norm.pdf(grid, x_bar, s_n), "k--", label=f"normal with the same mean and SD")
ax.set_xlabel("log2 fold change (tumour vs normal)"); ax.set_ylabel("density")
ax.set_title(f"SIMULATED: is it normally distributed?  X̄ = {x_bar:.2f}, s = {s_n:.2f}"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 100 values have sample average $\bar X_n = 0.40$ and sample standard deviation
$s_n = 1.62$. One crude check even looks reassuring — 69 % of the values lie within one SD of the mean, as for a normal
distribution (68 %) — but the histogram is lopsided: nothing below −2.6, where the normal curve still expects values, and
a long right tail reaching 5.7 (skewness 0.91; a normal distribution has 0). Describing the data (phase 2) already warns
us that a method assuming normal data may be unsafe. Notice that $\bar X_n$ and $s_n$ alone could never have told us
this: very different shapes share the same mean and SD.

### What is a statistic?

**Definition.** A **statistic** is anything that can be computed from the collected data — it must be **observable**.
$\bar X_n$, $s_n$, the median, the largest value, the fraction of heads are statistics. The true mean $\mu$ of the
tissue is *not* a statistic: nobody can compute it from 30 cells; it is a **parameter** of the distribution, which we
try to *estimate* with a statistic.

* A **point statistic** is a single value computed from data (e.g. $\bar X_n$, $s_n$).
* An **interval (or range) statistic** is an interval $[a, b]$ computed from the data (e.g. the range
  $[\min_i X_i, \max_i X_i]$, or the confidence interval of Section 7).

**A statistic is itself a random variable**, because a new experiment produces new data, and new data give a new value.
Let us repeat both experiments a few times.

In [ ]:
# ── 1.3 A statistic is a random variable: repeat each experiment five times (SIMULATED) ──
rng_rep = np.random.default_rng(SEED + 13)
print("five new experiments of 100 fair-coin flips → X̄_100:",
      [f"{rng_rep.binomial(1, 0.5, 100).mean():.2f}" for _ in range(5)])
print("five new experiments of 30 cells → (X̄_30, s_30, range):")
for _ in range(5):
    c = simulate_mrna_per_cell(30, rng_rep)
    print(f"   X̄ = {c.mean():5.2f}   s = {c.std(ddof=1):5.2f}   range = [{c.min()}, {c.max()}]")

🔍 **What just happened?** Five repetitions of the fair-coin experiment gave five different sample proportions
(0.57, 0.52, 0.45, 0.40, 0.49), and five new sets of 30 cells gave five different means, SDs and ranges. The parameters (0.5 for the coin;
$\mu = 6$, $\sigma = 5.48$ for the cells) never changed — only the data did. The distribution of a statistic over
imaginary repetitions of the experiment is called its **sampling distribution** (Section 6); its spread tells us how much
to trust one value.

✏️ **Your turn.** (a) Is the number of cells with zero molecules a statistic? Is "the probability that a cell has zero
molecules"? (b) Rerun cell 1.3 with 1,000 cells per experiment instead of 30. Which statistics become more stable, and
which (hint: the range) keep growing?

### 📘 Lecture notes — statistics in a nutshell; what a statistic is

**Statistics** deals with data; its goal is to make **inferences based on data**. Three overlapping phases:
(1) **collecting data** in experiments, *preceded by forming hypotheses* about the phenomena of interest;
(2) **describing data** — describe the results; (3) **analysing data** — infer from the results the strength of the
evidence with respect to the hypotheses.

**Two guiding questions.** "Is it fair?" — 100 flips coded $X_i = 1$ (H), $0$ (T), summarised by $\bar X_{100}$
(the record of Section 1.1 has 52 heads, so $\bar X_{100} = 0.52$; 48 is the number of tails). "Is it normally
distributed?" — a histogram summarised by the sample average $\bar X_n$ and the sample standard deviation $s_n$
(our simulated example: $\bar X_n \approx$ 0.40, $s_n \approx$ 1.62, visibly skewed).

**Definition.** A **statistic** is anything that can be computed from the collected data (it must be *observable*).
*Point statistic*: a single value ($\bar X_n$, $s_n$). *Interval or range statistic*: an interval $[a, b]$ computed from
the data. **A statistic is itself a random variable**, because a new experiment produces new data to compute it from.
A **parameter** ($\mu$, $\sigma$, $p$) describes the distribution and is not observable; statistics *estimate* parameters.

> 🎓 **Exam tip.** If you are asked whether something is a statistic, ask: "could I compute it from the data alone,
> without knowing the true distribution?" $\bar X_n$: yes. $\mu$, or $\bar X_n - \mu$: no.

---
## 2 · i.i.d. data and the sample mean as a random variable

> 📍 **Where we are.** The most important statistic is the average. Using only the rules for expectation and variance
> from Module 21, we can say exactly where the average is centred and how much it wobbles — the two facts behind every
> theorem in this module.

**i.i.d. data.** We model a data set $X_1, X_2, \dots, X_n$ as **independent and identically distributed** (i.i.d.)
random variables: *identically distributed* — every $X_i$ comes from the same unknown distribution $P(X)$, so they all
have the same mean $\mu = E[X_i]$ and the same variance $\sigma^2 = \operatorname{Var}(X_i)$; *independent* — knowing
some of the values tells you nothing about the others (Module 19). Thirty cells picked from different places in a
tissue are close to i.i.d.; thirty measurements of the *same* cell are not identically distributed draws from the tissue.

**The sample mean.** Let $\bar X_n$ be the average of the observations:

$$\bar X_n = \frac{X_1 + X_2 + \dots + X_n}{n} = \frac{1}{n}\sum_{i=1}^{n} X_i .$$

*In words:* add up the $n$ observed values and divide by how many there are. Note two things. $\bar X_n$ is **different
from $\mu$**, the expected value of the underlying distribution: $\mu$ is a fixed (unknown) number, while $\bar X_n$
depends on which cells happened to be sampled. And $\bar X_n$ is **itself a random variable**.

**Example.** 100 fair coin flips $C_1, \dots, C_{100}$, i.i.d., with value 1 for heads and 0 for tails. If 48 of them are
heads, then $\bar C_{100} = 48/100 = 0.48$, although $\mu = E[C_i] = 0.5$.

🧮 **Math refresher — the rules we need from Module 21.** For random variables $X, Y$ and constants $a, b$:
(i) $E[aX + b] = aE[X] + b$; (ii) $E[X + Y] = E[X] + E[Y]$ — *always*, independent or not; (iii)
$\operatorname{Var}(aX + b) = a^2\operatorname{Var}(X)$ — constants shift without spreading, and scaling by $a$ scales the
variance by $a^2$; (iv) $\operatorname{Var}(X + Y) = \operatorname{Var}(X) + \operatorname{Var}(Y)$ — *only if* $X$ and $Y$
are independent (otherwise add $2\operatorname{Cov}(X, Y)$). All four extend to sums of $n$ terms.

**The mean of the mean.** Using (i) with $a = 1/n$ and then (ii):

$$E[\bar X_n] = E\Big[\frac1n\sum_{i=1}^n X_i\Big] = \frac1n\,E\Big[\sum_{i=1}^n X_i\Big] = \frac1n\sum_{i=1}^n E[X_i]
= \frac1n\cdot n\mu = \mu .$$

*In words:* on average over many experiments, the sample mean hits the true mean — it is an **unbiased** estimator of
$\mu$. No independence was needed for this line.

**The variance of the mean.** Using (iii) with $a = 1/n$ and then (iv), which needs independence:

$$\operatorname{Var}(\bar X_n) = \operatorname{Var}\Big(\frac1n\sum_{i=1}^n X_i\Big) = \frac{1}{n^2}\operatorname{Var}\Big(\sum_{i=1}^n X_i\Big)
= \frac{1}{n^2}\sum_{i=1}^n \operatorname{Var}(X_i) = \frac{1}{n^2}\cdot n\sigma^2 = \frac{\sigma^2}{n} .$$

*In words:* averaging $n$ independent values divides the variance by $n$. Its square root,

$$\operatorname{SD}(\bar X_n) = \frac{\sigma}{\sqrt n}, \qquad\text{called the **standard error** of the mean (SE)},$$

*in words:* the typical distance between a sample mean and the true mean shrinks like one over the square root of the
sample size — to halve the error you need **four times** as many cells.

The same rules give the **sum** $S_n = X_1 + \dots + X_n = n\bar X_n$: $E[S_n] = n\mu$, $\operatorname{Var}(S_n) = n\sigma^2$,
$\operatorname{SD}(S_n) = \sqrt n\,\sigma$. The sum spreads out as $n$ grows, the mean concentrates.

In [ ]:
# ── 2.1 Check E[X̄] = μ and Var(X̄) = σ²/n with 100,000 simulated experiments of n = 30 cells ──
n_cells, n_experiments = 30, 100_000
rng_sim = np.random.default_rng(SEED + 21)
many = simulate_mrna_per_cell(n_cells * n_experiments, rng_sim).reshape(n_experiments, n_cells)
means = many.mean(axis=1)                         # one X̄_30 per experiment
sums = many.sum(axis=1)                           # one S_30 per experiment
print(f"{'':22s}{'by formula':>14s}{'by simulation':>16s}")
print(f"{'E[X̄_30]':22s}{TRUE_MEAN:14.4f}{means.mean():16.4f}")
print(f"{'Var(X̄_30) = σ²/n':22s}{TRUE_VAR / n_cells:14.4f}{means.var():16.4f}")
print(f"{'SE = σ/√n':22s}{TRUE_SD / sqrt(n_cells):14.4f}{means.std():16.4f}")
print(f"{'E[S_30] = nμ':22s}{n_cells * TRUE_MEAN:14.1f}{sums.mean():16.1f}")
print(f"{'SD(S_30) = √n σ':22s}{sqrt(n_cells) * TRUE_SD:14.3f}{sums.std():16.3f}")

🔍 **What just happened?** Across 100,000 simulated experiments the average of the 30-cell means is 6.0002 (formula: 6),
their variance 1.0019 (formula: $30/30 = 1.0$) and their standard deviation 1.0010 (formula: $\sqrt{30}/\sqrt{30} = 1.0$).
So one experiment's mean is typically about 1 molecule away from the truth — even though single cells vary by $\sigma
\approx 5.5$. The sums behave the opposite way: centred at 180 with SD $\approx 30$, they spread *more* than single cells.

In [ ]:
# ── 2.2 The standard error shrinks like 1/√n ──
sizes = np.array([1, 2, 5, 10, 30, 100, 300, 1000])
rng_se = np.random.default_rng(SEED + 22)
sd_of_means = [simulate_mrna_per_cell(n * 4000, rng_se).reshape(4000, n).mean(axis=1).std() for n in sizes]
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.loglog(sizes, TRUE_SD / np.sqrt(sizes), "k-", label="σ/√n (formula)")
ax.loglog(sizes, sd_of_means, "o", color="tab:orange", label="SD of 4,000 simulated means")
for n_, s_ in zip(sizes, sd_of_means):
    ax.annotate(f"{s_:.2f}", (n_, s_), textcoords="offset points", xytext=(4, 4), fontsize=7)
ax.set_yticks([0.2, 0.5, 1, 2, 5]); ax.set_yticklabels(["0.2", "0.5", "1", "2", "5"])
ax.yaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel("number of cells averaged, n (log scale)"); ax.set_ylabel("standard error of X̄_n (log scale)")
ax.set_title("SIMULATED mRNA counts: the standard error falls with slope −1/2 on a log–log plot"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** On log–log axes $\sigma/\sqrt n$ is a straight line of slope $-1/2$, and the simulated standard
errors sit on it: 5.33 for one cell, 1.02 for 30 cells, 0.18 for 1,000. Going from 30 to 300 cells (ten times the
work) shrinks the error by only $\sqrt{10} \approx 3.2$. This "square-root law" is why precision is expensive.

✏️ **Your turn.** (a) By hand: a gene's expression has $\sigma = 0.8$ log2 units between mice. What is the standard error of
the mean of 4 mice? Of 16? (b) The derivation of $\operatorname{Var}(\bar X_n)$ used independence. Suppose the 30 cells
come in 15 pairs of sister cells with identical counts. What is $\operatorname{Var}(\bar X_{30})$ now? (Answers in the
✅ Checkpoint.)

---
## 3 · The law of large numbers

> 📍 **Where we are.** Section 2 showed that $\bar X_n$ is centred on $\mu$ with a spread that shrinks to zero. The law
> of large numbers turns this into a statement about a single long experiment: keep collecting data, and the running
> average settles down at the truth.

**Law of large numbers (LLN).** Let $X_1, \dots, X_n$ be i.i.d. with mean $\mu$. As $n$ grows, the probability that
$\bar X_n$ is close to $\mu$ approaches 1.

Below are running averages $\bar X_1, \bar X_2, \dots, \bar X_{10\,000}$ for four kinds of biological data: a fair coin
(a 1:1 Mendelian test cross), a die (an unbiased six-way choice), mRNA counts per cell (our bursty gene), and
exponential waiting times between mutations along a lineage (mean 2 time units).

In [ ]:
# ── 3.1 Running means settle at the true mean (four SIMULATED data streams) ──
def running_mean(x):
    """The sequence X̄_1, X̄_2, …, X̄_n of averages of the first k values, k = 1 … n."""
    x = np.asarray(x, dtype=float)
    return np.cumsum(x) / np.arange(1, len(x) + 1)


N_RUN = 10_000
rng_lln = np.random.default_rng(SEED + 31)
streams = {"fair coin / test cross (μ = 0.5)": (rng_lln.binomial(1, 0.5, N_RUN), 0.5),
           "die (μ = 3.5)": (rng_lln.integers(1, 7, N_RUN), 3.5),
           "mRNA molecules per cell (μ = 6)": (simulate_mrna_per_cell(N_RUN, rng_lln), TRUE_MEAN),
           "waiting time between mutations (μ = 2)": (rng_lln.exponential(2.0, N_RUN), 2.0)}
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6))
k = np.arange(1, N_RUN + 1)
for ax, (name, (x, mu)) in zip(axes.flat, streams.items()):
    rm = running_mean(x)
    ax.semilogx(k, rm, color="tab:blue", lw=1)
    ax.axhline(mu, color="tab:red", ls="--", lw=1, label="true mean μ")
    ax.set_title(name); ax.set_xlabel("number of observations n (log scale)"); ax.set_ylabel("running mean X̄_n")
    ax.legend(loc="best")
    print(f"{name:42s} X̄_10 = {rm[9]:6.3f}   X̄_100 = {rm[99]:6.3f}   X̄_10000 = {rm[-1]:6.3f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every running mean starts erratically — after 10 observations the four streams are at
0.400, 2.400, 9.300 and 1.318 — and then settles near its true mean: after 10,000 observations the values are 0.505, 3.507, 6.036 and 1.998 (true: 0.5, 3.5, 6,
2). The bursty mRNA counts and the skewed waiting times take longer to settle than the coin, because their $\sigma$ is
larger (5.48 and 2, against 0.5 for the coin); the speed is governed by $\sigma/\sqrt n$.

### 3.2 What "close" means: the precise statement

It is tempting to write the LLN as $\lim_{n\to\infty} P(\bar X_n = \mu) = 1$. **That is wrong.** For continuous data
$P(\bar X_n = \mu) = 0$ for every $n$ (a continuous random variable hits any single value with probability 0), and the
limit of zeros is 0. Even for a fair coin, where $\bar X_n = 1/2$ is possible when $n$ is even, the probability of
*exactly* one half shrinks to 0. The correct statement says "within $\varepsilon$":

$$\lim_{n\to\infty} P\big(|\bar X_n - \mu| < \varepsilon\big) = 1 \qquad\text{for every } \varepsilon > 0 .$$

*In words:* pick any tolerance $\varepsilon$, however small (0.05, 0.001, …); the probability that the sample mean lies
within that tolerance of the true mean tends to 1 as the sample grows. (This is the **weak** law of large numbers;
the *strong* law, which says the running mean converges with probability 1, is beyond this course.)

🧮 **Math refresher — absolute values as distances.** $|a - b|$ is the distance between $a$ and $b$ on the number line,
so $|\bar X_n - \mu| < \varepsilon$ is the same as $\mu - \varepsilon < \bar X_n < \mu + \varepsilon$: "$\bar X_n$ lies
in the window of half-width $\varepsilon$ around $\mu$".

Let us compute both probabilities exactly for a fair coin, where $n\bar X_n \sim \text{Binomial}(n, 1/2)$ (Module 18).

In [ ]:
# ── 3.2 P(X̄_n = 1/2) → 0, but P(|X̄_n − 1/2| < 0.05) → 1 (exact binomial calculations) ──
rows = []
for n_ in [2, 10, 100, 1000, 10_000]:
    heads = np.arange(n_ + 1)
    pmf = stats.binom.pmf(heads, n_, 0.5)
    inside = np.abs(20 * heads - 10 * n_) < n_          # |k/n − 1/2| < 1/20, in exact integer arithmetic
    rows.append({"n": n_, "P(X̄_n = 0.5)": pmf[n_ // 2], "P(|X̄_n − 0.5| < 0.05)": pmf[inside].sum()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

🔍 **What just happened?** The probability of getting *exactly* half heads falls from 0.5 ($n = 2$) to 0.0796
($n = 100$) to 0.0080 ($n = 10\,000$) — it goes to 0, roughly like $\sqrt{2/(\pi n)}$. Meanwhile the probability of
landing *within 0.05* of one half rises from 0.25 ($n = 10$) to 0.63 ($n = 100$), 0.998 ($n = 1000$) and 1.0000
($n = 10\,000$). The sample mean is almost never *equal* to $\mu$, but it is almost always *close* to it. This is the
"pedantry detail" that matters: a theorem stated with "$=$" would be false.

### 3.3 Why the LLN is true: Chebyshev's inequality

🧮 **Math refresher — Markov's inequality.** If $Y \ge 0$ and $a > 0$ then $P(Y \ge a) \le E[Y]/a$. *Proof:* $Y$ is at
least $a$ on the event $\{Y \ge a\}$ and at least 0 elsewhere, so $E[Y] \ge a\cdot P(Y \ge a) + 0$; divide by $a$.
(Example: if the mean litter size is 6, at most $6/12 = 50\,\%$ of litters can have 12 or more pups.)

Apply Markov's inequality to the non-negative variable $Y = (\bar X_n - \mu)^2$ with $a = \varepsilon^2$. Because
$|\bar X_n - \mu| \ge \varepsilon$ exactly when $(\bar X_n - \mu)^2 \ge \varepsilon^2$,

$$P\big(|\bar X_n - \mu| \ge \varepsilon\big) = P\big((\bar X_n - \mu)^2 \ge \varepsilon^2\big) \le
\frac{E[(\bar X_n - \mu)^2]}{\varepsilon^2} = \frac{\operatorname{Var}(\bar X_n)}{\varepsilon^2} = \frac{\sigma^2}{n\varepsilon^2} .$$

*In words:* this is **Chebyshev's inequality** for the sample mean. The chance of missing $\mu$ by $\varepsilon$ or more is
at most $\sigma^2/(n\varepsilon^2)$, which tends to 0 as $n \to \infty$ for any fixed $\varepsilon$ — so
$P(|\bar X_n - \mu| < \varepsilon) \ge 1 - \sigma^2/(n\varepsilon^2) \to 1$. That *is* the law of large numbers, proved in
one line from $\operatorname{Var}(\bar X_n) = \sigma^2/n$ — provided $\sigma^2$ is finite.

In [ ]:
# ── 3.3 Chebyshev's bound vs the exact miss probability for a fair coin (σ² = 1/4, ε = 0.05) ──
rows = []
for n_ in [10, 100, 400, 1000, 10_000]:
    heads = np.arange(n_ + 1)
    pmf = stats.binom.pmf(heads, n_, 0.5)
    miss = pmf[np.abs(20 * heads - 10 * n_) >= n_].sum()       # P(|X̄_n − 1/2| ≥ 0.05), exact
    rows.append({"n": n_, "exact P(miss by ≥ 0.05)": miss, "Chebyshev bound σ²/(nε²)": min(1.0, 0.25 / (n_ * 0.05 ** 2))})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.2e}" if v < 1e-3 else f"{v:.4f}"))

🔍 **What just happened?** The bound $\sigma^2/(n\varepsilon^2) = 100/n$ is always above the exact probability, as it
must be, but it is very conservative: at $n = 1000$ it guarantees a miss probability of at most 0.1, while the true value
is 0.0017. Chebyshev needs only the variance, so it holds for *every* distribution — and pays for that generality.
The central limit theorem (Section 4) gives much sharper approximations.

### 3.4 When averaging does not help: the Cauchy warning

The proof needed a finite variance (and the theorem a finite mean). Some distributions have neither. The classic one
is the **Cauchy** distribution — the distribution of the ratio $Z_1/Z_2$ of two independent standard normals. It arises
whenever you divide by a noisy quantity that can be close to zero, for example a fold change computed as the ratio of two
background-subtracted intensities that are both near zero.

In [ ]:
# ── 3.4 The running mean of Cauchy data never settles (SIMULATED ratios of two noisy measurements) ──
rng_c = np.random.default_rng(SEED + 34)
N_C = 100_000
ratio = rng_c.standard_normal(N_C) / rng_c.standard_normal(N_C)        # a standard Cauchy sample
normal_ref = rng_c.standard_normal(N_C)
kk = np.arange(1, N_C + 1)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogx(kk, running_mean(normal_ref), color="tab:blue", lw=1, label="normal data: settles at 0")
ax.semilogx(kk, running_mean(ratio), color="tab:red", lw=1, label="Cauchy data (ratio of two normals): never settles")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("number of observations n (log scale)"); ax.set_ylabel("running mean")
ax.set_title("SIMULATED: the law of large numbers needs a finite mean"); ax.legend()
plt.tight_layout(); plt.show()
print("running mean of the Cauchy data at n = 10³, 10⁴, 10⁵:",
      [f"{running_mean(ratio)[m - 1]:.2f}" for m in (1_000, 10_000, 100_000)])
print("largest |ratio| observed:", f"{np.abs(ratio).max():.0f}")

🔍 **What just happened?** The running mean of the normal data hugs 0, but the Cauchy running mean keeps jumping —
−1.84, 0.26 and 3.02 at $n = 10^3, 10^4, 10^5$ — because now and then a denominator close to zero produces a gigantic value (the
largest here is about 150,000), which drags the average away. For the Cauchy distribution the mean of $n$ values has
*exactly the same* Cauchy distribution as a single value: averaging buys nothing. Lesson for the bench: averaging ratios
with noisy denominators can be meaningless; average on a log scale, or average numerator and denominator separately.

✏️ **Your turn.** (a) Use Chebyshev to find an $n$ that guarantees $P(|\bar X_n - 6| \ge 1) \le 0.05$ for our mRNA counts
($\sigma^2 = 30$). (b) Replace the Cauchy ratio by `rng_c.standard_normal(N_C) / (3 + rng_c.standard_normal(N_C))`.
Does the running mean settle now? Why does a denominator that stays away from zero help?

### 📘 Lecture notes — i.i.d. data, the sample mean and the law of large numbers

**Set-up.** A data set $X_1, \dots, X_n$ of **i.i.d.** (independent and identically distributed) random variables from
the same unknown distribution $P(X)$: every $X_i$ has the same $\mu$ and $\sigma^2$.

**Sample mean.** $\bar X_n = \frac{X_1 + \dots + X_n}{n} = \frac1n\sum_{i=1}^n X_i$ — different from the expected value $\mu$
of the underlying distribution, and **itself a random variable**. *Example:* 100 fair flips with 48 heads give
$\bar C_{100} = 48/100$, although $\mu = 1/2$.

**Mean and variance of the mean** (Module 21 rules):

$$E[\bar X_n] = \mu,\qquad \operatorname{Var}(\bar X_n) = \frac{\sigma^2}{n},\qquad \operatorname{SE} = \frac{\sigma}{\sqrt n};
\qquad E[S_n] = n\mu,\quad \operatorname{Var}(S_n) = n\sigma^2 .$$

*In words:* the mean is unbiased (linearity, no independence needed) and its variance falls like $1/n$ (needs independence).

**Law of large numbers.** As $n$ grows, the probability that $\bar X_n$ is close to $\mu$ goes to 1:
$\lim_{n\to\infty} P(|\bar X_n - \mu| < \varepsilon) = 1$ for every $\varepsilon > 0$.
⚠️ Not "$P(\bar X_n = \mu) \to 1$": for continuous data that probability is 0 for every $n$, and for a fair coin
$P(\bar X_n = 1/2) \to 0$ (0.0796 at $n = 100$, 0.0080 at $n = 10\,000$).

**Proof (Chebyshev).** $P(|\bar X_n - \mu| \ge \varepsilon) \le \sigma^2/(n\varepsilon^2) \to 0$ (Markov's inequality applied
to $(\bar X_n - \mu)^2$). Needs a finite variance; the Cauchy distribution (ratio of two normals) has no mean and its
running average never settles.

> 🎓 **Exam tip.** Be ready to derive $\operatorname{Var}(\bar X_n) = \sigma^2/n$ line by line and to say where
> independence is used (only in the variance step). If asked to "state the LLN", include the $\varepsilon$.

---
## 4 · The central limit theorem

> 📍 **Where we are.** The LLN says $\bar X_n$ ends up near $\mu$. It does not say *how* the remaining error is
> distributed. The central limit theorem (CLT) does — and the answer is the same bell curve whatever the data look like.

🧮 **Math refresher — the normal distribution and standardising.** $\mathcal N(\mu, \sigma^2)$ denotes the normal (Gaussian)
distribution with mean $\mu$ and **variance** $\sigma^2$ (Module 18). If $Y \sim \mathcal N(m, v)$ then
$Z = (Y - m)/\sqrt v \sim \mathcal N(0, 1)$, the **standard normal**, whose CDF is written $\Phi(z) = P(Z \le z)$ and computed
with `stats.norm.cdf(z)`. Standardising means "subtract the mean, divide by the standard deviation": it measures a value
in units of standard deviations from its mean (a z-score). About 68 % / 95 % / 99.7 % of a normal lies within 1 / 2 / 3
standard deviations of its mean.

**Set-up.** $X_1, \dots, X_n$ i.i.d. with mean $\mu = E[X_i]$ and **finite** standard deviation $\sigma$
($\operatorname{Var}(X_i) = \sigma^2$). Let $S_n = X_1 + \dots + X_n$ and $\bar X_n = S_n/n$. From Section 2,
$E[S_n] = n\mu$, $\operatorname{SD}(S_n) = \sqrt n\,\sigma$, $E[\bar X_n] = \mu$, $\operatorname{SD}(\bar X_n) = \sigma/\sqrt n$.

**Standardisation.** Since $S_n$ and $\bar X_n$ are multiples of each other, they have the same standardisation: multiply
numerator and denominator of the second fraction by $n$,

$$Z_n = \frac{\bar X_n - \mu}{\sigma/\sqrt n} = \frac{n\bar X_n - n\mu}{n\sigma/\sqrt n} = \frac{S_n - n\mu}{\sigma\sqrt n} .$$

*In words:* $Z_n$ measures how many standard errors the sample mean lies from the true mean — and it is the same number
whether you look at the mean or at the sum.

**Central limit theorem.** As $n \to \infty$, the distribution of $Z_n$ converges to the standard normal: for every $z$,

$$\lim_{n\to\infty} P(Z_n \le z) = \Phi(z) .$$

*In words:* whatever the shape of the data's distribution (as long as its variance is finite), the standardised mean
behaves more and more like a standard normal variable. For large $n$ this gives the working approximations

$$\bar X_n \approx \mathcal N\Big(\mu, \frac{\sigma^2}{n}\Big), \qquad S_n \approx \mathcal N(n\mu,\ n\sigma^2), \qquad Z_n \approx \mathcal N(0, 1) .$$

*In words:* the sample mean is approximately normal, centred at $\mu$, with variance $\sigma^2/n$ (standard deviation
$\sigma/\sqrt n$); the sum is approximately normal with mean $n\mu$ and variance $n\sigma^2$.

> ⚠️ **Common confusion — the second parameter.** You will often see the CLT written $\bar X_n \approx N(\mu, \sigma/\sqrt n)$.
> That is the same statement **only if** the second slot means the *standard deviation*. In this course (and in
> `stats.norm(loc, scale)`, where `scale` is the SD!) be explicit: variance $\sigma^2/n$, standard deviation
> $\sigma/\sqrt n$. Likewise $S_n \approx N(n\mu, \sqrt n\sigma)$ uses the SD convention. Also, a limit cannot depend on
> $n$: "$\lim_{n\to\infty}\bar X_n \sim N(\mu, \sigma^2/n)$" is not meaningful — the honest limit statement is the one
> for $Z_n$ above, and "$\approx$ for large $n$" is the practical reading.

Let us see the theorem happen. We take three skewed or discrete distributions, draw 20,000 experiments of size
$n = 1, 2, 5, 30$, standardise each mean, and compare the histogram of $Z_n$ with the standard normal curve.

In [ ]:
# ── 4.1 The CLT in pictures: standardised means of skewed data at n = 1, 2, 5, 30 (SIMULATED) ──
def plot_standardised_means(ax, z, lattice_step=None, bins=40):
    """Histogram (as a density) of standardised means z; for lattice data (integer counts) one bar per possible value,
    with bar area = probability, so that it is comparable with the N(0, 1) density."""
    if lattice_step is not None:
        vals, counts = np.unique(np.round(z, 9), return_counts=True)       # the possible values of Z_n
        ax.bar(vals, counts / len(z) / lattice_step, width=lattice_step * 0.9, color="tab:blue", alpha=0.6)
    else:
        ax.hist(z, bins=bins, range=(-4, 6), density=True, color="tab:blue", alpha=0.6)
    g = np.linspace(-4, 6, 300)
    ax.plot(g, stats.norm.pdf(g), "k-", lw=1.2)
    ax.set_xlim(-4, 6)


rng_clt = np.random.default_rng(SEED + 41)
n_exp = 20_000
families = [("exponential waiting time (mean 1)", lambda size: rng_clt.exponential(1.0, size), 1.0, 1.0, False),
            ("Poisson mRNA count (λ = 1)", lambda size: rng_clt.poisson(1.0, size), 1.0, 1.0, True),
            ("Bernoulli carrier (p = 0.1)", lambda size: rng_clt.binomial(1, 0.1, size), 0.1, sqrt(0.1 * 0.9), True)]
ns = [1, 2, 5, 30]
fig, axes = plt.subplots(3, 4, figsize=(10, 6.6), sharex=True)
for r, (name, draw, mu, sd, lattice) in enumerate(families):
    for c, n_ in enumerate(ns):
        xbar = draw((n_exp, n_)).mean(axis=1)
        z = (xbar - mu) / (sd / np.sqrt(n_))
        plot_standardised_means(axes[r, c], z, lattice_step=(1 / (sd * np.sqrt(n_)) if lattice else None))
        axes[r, c].set_title(f"n = {n_}" + (f"\n{name}" if c == 0 else ""), fontsize=8, loc="left")
        if r == 2:
            axes[r, c].set_xlabel("standardised mean Z_n")
        print(f"{name:36s} n = {n_:2d}: skewness of Z_n = {stats.skew(z):5.2f}, "
              f"P(Z_n ≤ 1.96) = {np.mean(z <= 1.96):.3f} (normal: 0.975)")
fig.suptitle("SIMULATED: histograms of 20,000 standardised means vs the N(0, 1) curve (black)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read each row from left to right. With $n = 1$ we see the data's own distribution: the
exponential is a steep ramp, the Poisson(1) count is a few bars (0, 1, 2, … molecules), the Bernoulli(0.1) has just two
values (each bar's *area* is the probability of that value). Averaging
pulls them towards the bell: the skewness of $Z_n$ for the exponential falls from 1.98 ($n = 1$) to 0.37 ($n = 30$),
close to the theoretical $2/\sqrt n$. The Bernoulli(0.1) is the slowest — at $n = 30$ it is still visibly skewed and
lumpy (skewness 0.49), because an event with probability 0.1 happens only 3 times in 30 trials on average. **The CLT
always arrives, but rare events and skewed data make it arrive late.** Note also that discrete data stay discrete: the CLT
is about the *shape* of the histogram (its CDF), not about the values becoming continuous.

### 4.2 Biology: why polygenic traits are bell-shaped

Height, birth weight, LDL cholesterol and many other quantitative traits are influenced by hundreds or thousands of
genetic variants, each with a small effect, plus many small environmental influences. A trait value is therefore a **sum**
of many independent small contributions — exactly the situation of the CLT. This is how R. A. Fisher (1918) reconciled
Mendel's discrete genes with the continuous, bell-shaped variation measured by biometricians. Let us build a trait from
1, 2, 10 and 200 SNPs (SIMULATED, additive effects, Hardy–Weinberg genotypes from the toolkit).

In [ ]:
# ── 4.2 A polygenic trait as a sum of SNP effects (SIMULATED genotypes and effect sizes) ──
n_people, n_snps = 5000, 200
rng_pg = np.random.default_rng(SEED + 42)
allele_freqs = rng_pg.uniform(0.1, 0.5, n_snps)
effects = rng_pg.normal(0.0, 1.0, n_snps)                   # effect of each copy of the allele (trait units)
G = np.column_stack([simulate_genotypes(n_people, allele_freqs[j], seed=1000 + j) for j in range(n_snps)])
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, k_snps in zip(axes, [1, 2, 10, 200]):
    trait = G[:, :k_snps] @ effects[:k_snps]                  # additive genetic value from the first k SNPs
    z = (trait - trait.mean()) / trait.std()
    ax.hist(z, bins=40, range=(-4, 4), density=True, color="tab:green", alpha=0.6)
    g = np.linspace(-4, 4, 200); ax.plot(g, stats.norm.pdf(g), "k-", lw=1)
    ax.set_title(f"{k_snps} SNP{'s' if k_snps > 1 else ''}: {len(np.unique(trait))} distinct values", fontsize=8)
    ax.set_xlabel("standardised trait"); ax.set_yticks([])
    print(f"{k_snps:3d} SNPs: distinct trait values = {len(np.unique(trait)):5d}, skewness = {stats.skew(trait):5.2f}")
fig.suptitle("SIMULATED: a trait that sums many small genetic effects becomes bell-shaped", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With one SNP the "trait" takes only 3 values (genotypes 0, 1, 2) — three spikes, like a
Mendelian trait. Two SNPs give 9 combinations; ten SNPs give 3,492 distinct values and a histogram that is already
bell-like but still lopsided (skewness −0.32); 200 SNPs give a smooth bell with skewness 0.01, although no single SNP is normally distributed. Add many small environmental effects and the bell gets
even smoother. That is why quantitative-genetics models (and the linear regression of Module 27) can treat traits as
normal.

✏️ **Your turn.** (a) Make one SNP a "major gene": multiply `effects[0]` by 20 before computing the 200-SNP trait. What
does the histogram look like now, and which assumption of the CLT (identically distributed terms, or no single term
dominating) is violated? (b) Change the Bernoulli row of cell 4.1 to $p = 0.5$. How fast does it look normal now?

### 📘 Lecture notes — the central limit theorem

**Statement.** $X_1, \dots, X_n$ i.i.d. with mean $\mu$ ($E[X_i] = \mu$) and standard deviation $\sigma$
($\operatorname{Var}(X_i) = \sigma^2 < \infty$). Let $S_n = \sum_i X_i$ and $\bar X_n = S_n/n$. The properties of mean and variance give
$E(S_n) = n\mu$, $\operatorname{Var}(S_n) = n\sigma^2$, $\sigma_{S_n} = \sqrt n\sigma$; $E(\bar X_n) = \mu$,
$\operatorname{Var}(\bar X_n) = \sigma^2/n$, $\sigma_{\bar X_n} = \sigma/\sqrt n$. Since $S_n$ and $\bar X_n$ are multiples of
each other they have the same standardisation

$$Z_n = \frac{S_n - n\mu}{\sigma\sqrt n} = \frac{\bar X_n - \mu}{\sigma/\sqrt n} .$$

**CLT:** $P(Z_n \le z) \to \Phi(z)$ as $n \to \infty$. For large $n$:
$\bar X_n \approx \mathcal N(\mu, \sigma^2/n)$, $S_n \approx \mathcal N(n\mu, n\sigma^2)$, $Z_n \approx \mathcal N(0, 1)$.

*In words:* the distribution of the sample mean converges (after standardising) to the normal, whatever the data's distribution,
provided the variance is finite. LLN and CLT both say that more data make the empirical mean approach the underlying
(theoretical) mean; the CLT is more precise — it describes the *size* of the remaining error, so it lets us approximate
the probability that the sample size is not adequate.

**Common confusions.** (1) Second parameter: $\mathcal N(\mu, \sigma^2/n)$ in variance notation $=$ "$N(\mu, \sigma/\sqrt n)$" in
SD notation — say which. (2) The CLT is about the *mean/sum*, not the data: the cells stay skewed; their average becomes normal.
(3) "Large $n$" depends on the skewness: $n = 30$ is plenty for a symmetric distribution, not for Bernoulli(0.1).
(4) Infinite variance (Cauchy) breaks it.

**Biology.** Polygenic traits are sums of many small effects → approximately normal (Fisher 1918).

> 🎓 **Exam tip.** Given $\mu$, $\sigma$ and $n$, you should be able to write down the approximate distribution of both
> $\bar X_n$ and $S_n$ (with variances!) and convert a question about either into a question about $Z \sim \mathcal N(0, 1)$.

---
## 5 · Why use the CLT? Approximate probabilities and adequate sample sizes

> 📍 **Where we are.** For coin flips we computed probabilities *exactly* with the binomial distribution. So what is the
> point of an *approximate* answer from the CLT?

Three reasons.

1. **In general the distribution of the sum is not familiar.** Each $X_i$ was Bernoulli, so the sum was binomial and we
   could look it up. For most data — bursty mRNA counts, a mixture of silent and active cells, measurement errors of
   unknown shape — the distribution of $S_n$ has no name and no formula.
2. **Exact computation may be possible in theory but too expensive in practice**, even for a computer: the PMF of a sum is a
   repeated *convolution* (a sum over all ways of splitting the total), whose cost grows with $n$ and with the range of values.
3. **The CLT applies when $X_i$ has almost any distribution** — all we need are $\mu$ and $\sigma$. The price: some
   distributions need a larger $n$ for the approximation to be good (Section 4.1).

**Recipe.** To approximate $P(S_n \ge s)$: standardise, $z = (s - n\mu)/(\sigma\sqrt n)$, and use $P(Z \ge z) = 1 - \Phi(z)$. For
integer-valued data, replacing $s$ by $s - \tfrac12$ (the **continuity correction**) treats each integer as the interval
$[s - \tfrac12, s + \tfrac12]$ and usually improves the approximation.

### 5.1 A case with an exact answer: at least 60 heads in 100 fair flips

In [ ]:
# ── 5.1 Exact binomial vs CLT for P(S_100 ≥ 60), fair coin ──
n_, p_ = 100, 0.5
mu_S, sd_S = n_ * p_, sqrt(n_ * p_ * (1 - p_))
exact = stats.binom.sf(59, n_, p_)                    # P(S ≥ 60) = 1 − P(S ≤ 59)
z_plain = (60 - mu_S) / sd_S
z_cc = (59.5 - mu_S) / sd_S
print(f"E[S] = {mu_S:.0f}, SD(S) = √(n p (1 − p)) = {sd_S:.0f}")
print(f"exact binomial:                  P(S ≥ 60) = {exact:.4f}")
print(f"CLT, z = (60 − 50)/5 = {z_plain:.2f}:        1 − Φ(z) = {stats.norm.sf(z_plain):.4f}")
print(f"CLT + continuity, z = {z_cc:.2f}:       1 − Φ(z) = {stats.norm.sf(z_cc):.4f}")

🔍 **What just happened?** The exact probability of 60 or more heads is 0.0284. The CLT gives 0.0228 with $z = 2$, and
0.0287 with the continuity correction ($z = 1.9$) — within 0.0003 of the exact value, using nothing but $\mu$, $\sigma$
and the normal table.

### 5.2 A case without a familiar answer: the total transcript count in 50 cells

Suppose 30 % of cells are transcriptionally silent (0 molecules) and the other 70 % carry a Poisson(6) number of
molecules — a "zero-inflated" Poisson, typical of genes that are switched off in part of a population. What is the
probability that 50 cells together contain 250 or more molecules? The distribution of the total has no name; we can
compute it exactly only by convolving the single-cell PMF with itself 50 times.

In [ ]:
# ── 5.2 Total molecules in 50 cells of a zero-inflated Poisson: exact convolution vs CLT vs simulation ──
p_silent, lam, n_cells50, threshold = 0.3, 6.0, 50, 250
support = np.arange(40)                                       # P(count ≥ 40) is negligible (< 10⁻¹⁵)
pmf_cell = (1 - p_silent) * stats.poisson.pmf(support, lam)
pmf_cell[0] += p_silent
mu_cell = (support * pmf_cell).sum()
var_cell = (support ** 2 * pmf_cell).sum() - mu_cell ** 2
pmf_total = np.array([1.0])
for _ in range(n_cells50):                                    # 50 convolutions: the PMF of the sum
    pmf_total = np.convolve(pmf_total, pmf_cell)
exact_tail = pmf_total[threshold:].sum()
z = (threshold - 0.5 - n_cells50 * mu_cell) / sqrt(n_cells50 * var_cell)
rng_zip = np.random.default_rng(SEED + 52)
sim_counts = np.where(rng_zip.random((100_000, n_cells50)) < p_silent, 0, rng_zip.poisson(lam, (100_000, n_cells50)))
print(f"one cell: μ = {mu_cell:.3f}, σ² = {var_cell:.3f}  →  total of 50: mean {n_cells50 * mu_cell:.1f}, "
      f"SD {sqrt(n_cells50 * var_cell):.2f}")
print(f"P(total ≥ {threshold}): exact (50 convolutions) = {exact_tail:.4f} | CLT (z = {z:.3f}) = {stats.norm.sf(z):.4f} | "
      f"simulation of 100,000 tissues = {np.mean(sim_counts.sum(axis=1) >= threshold):.4f}")

🔍 **What just happened?** One cell has mean $0.7 \times 6 = 4.2$ and variance $0.7\,(6 + 36) - 4.2^2 = 11.76$, so 50 cells
total 210 molecules on average with SD $\sqrt{50 \times 11.76} = 24.25$. The exact tail probability from 50 convolutions
is 0.0526; the CLT gives 0.0517 from two numbers and one table lookup; brute-force simulation agrees (0.0521).
The CLT answer is off by less than 0.001 — good enough for any planning decision.

### 5.3 Is the sample size adequate?

The CLT also answers the question the LLN leaves open: *how large must $n$ be?* For a fair coin, how many flips are
needed so that $P(|\bar X_n - 0.5| < 0.05) \ge 0.95$? By the CLT, $\bar X_n \approx \mathcal N(0.5, 0.25/n)$, so we need
$0.05 \ge 1.96 \times 0.5/\sqrt n$, i.e. $n \ge (1.96 \times 0.5/0.05)^2 = 384.2$, so $n = 385$. Chebyshev, which knows
only the variance, demands $\sigma^2/(n\varepsilon^2) \le 0.05$, i.e. $n \ge 0.25/(0.0025 \times 0.05) = 2000$.

In [ ]:
# ── 5.3 How many flips for a 95 % chance of landing within 0.05 of 1/2? ──
eps, conf = 0.05, 0.95
z95 = stats.norm.ppf(1 - (1 - conf) / 2)
n_clt = int(np.ceil((z95 * 0.5 / eps) ** 2))
n_cheb = int(np.ceil(0.25 / (eps ** 2 * (1 - conf))))
for n_ in [100, n_clt, n_cheb]:
    heads = np.arange(n_ + 1)
    pmf = stats.binom.pmf(heads, n_, 0.5)
    print(f"n = {n_:5d}: exact P(|X̄_n − 0.5| < 0.05) = {pmf[np.abs(20 * heads - 10 * n_) < n_].sum():.4f}")
print(f"z for 95 % = {z95:.4f}; CLT says n ≥ {n_clt}; Chebyshev says n ≥ {n_cheb}")
z_record = (0.52 - 0.5) / (0.5 / sqrt(100))
p_far_exact = stats.binom.cdf(48, 100, 0.5) + stats.binom.sf(51, 100, 0.5)        # exact: at most 48 or at least 52 heads
print(f"\nthe typed-in record: X̄_100 = 0.52 is z = {z_record:.1f} standard errors from 0.5; for a fair coin "
      f"P(|X̄_100 − 0.5| ≥ 0.02) ≈ {2 * stats.norm.sf(z_record):.2f} (CLT), exactly {p_far_exact:.2f} (binomial)")

🔍 **What just happened?** With the CLT's $n = 385$ the exact probability of landing within 0.05 is 0.9474 —
essentially the 95 % we asked for; Chebyshev's $n = 2000$ overshoots (probability 1.0000), and 100 flips give only
0.63. Back to "Is it fair?": a fair coin produces a sample proportion at least 0.02 away from 0.5 in about
69 % of all 100-flip experiments by the CLT (76 % by the exact binomial calculation — the number of heads is a whole
number, so the continuous normal curve is slightly off), so 52 heads is no evidence of unfairness at all.

✏️ **Your turn.** (a) An allele has true frequency 0.5 in a population. How many chromosomes must you genotype to have a
95 % chance that the observed frequency is within 0.01 of the truth? (b) Repeat 5.2 for a threshold of 300 molecules.
Is the CLT still accurate that far into the tail (compare the relative error)?

---
## 6 · Sampling distributions and standard errors

> 📍 **Where we are.** "The distribution of a statistic over repeated experiments" deserves its own name, because every
> error bar in science is a summary of one. Here we look at three statistics, not just the mean.

The **sampling distribution** of a statistic $T = r(X_1, \dots, X_n)$ is its probability distribution when the data are
i.i.d. draws from $P(X)$. Its standard deviation is the **standard error** of $T$, $\operatorname{SE}(T)$. For the mean we
have a formula ($\sigma/\sqrt n$, and a normal shape by the CLT); for other statistics we can always *simulate* it when
we know the truth — and in Section 9 we will learn to approximate it when we don't.

In [ ]:
# ── 6.1 Sampling distributions of the mean, the median and the SD (normal data, n = 20, 20,000 experiments) ──
rng_sd = np.random.default_rng(SEED + 61)
mu_ex, sigma_ex, n_ex = 8.0, 0.6, 20          # log2 expression of a gene in 20 mice: μ = 8, σ = 0.6 (SIMULATED)
data = rng_sd.normal(mu_ex, sigma_ex, (20_000, n_ex))
stats_ = {"sample mean": data.mean(axis=1), "sample median": np.median(data, axis=1),
          "sample SD (ddof = 1)": data.std(axis=1, ddof=1)}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.0))
for ax, (name, values) in zip(axes, stats_.items()):
    ax.hist(values, bins=50, color="tab:purple", alpha=0.6, density=True)
    ax.set_title(f"{name}\nmean {values.mean():.3f}, SE {values.std():.3f}", fontsize=9)
    ax.set_xlabel("value of the statistic"); ax.set_yticks([])
    print(f"{name:22s}: centre (mean of 20,000 values) = {values.mean():.3f}, standard error = {values.std():.3f}")
fig.suptitle("SIMULATED: three sampling distributions from the same 20,000 experiments", fontsize=10)
plt.tight_layout(); plt.show()
print(f"σ/√n = {sigma_ex / sqrt(n_ex):.4f}; large-sample SE of the median for normal data ≈ 1.2533 σ/√n = "
      f"{1.2533 * sigma_ex / sqrt(n_ex):.4f}")

🔍 **What just happened?** The same 20,000 experiments give three different sampling distributions. The mean is centred
at 7.999 with SE 0.136 (formula $0.6/\sqrt{20} = 0.134$). The median is also centred at 8 but wobbles more (SE
0.162, a little below the large-sample value $1.2533\,\sigma/\sqrt n = 0.168$ that holds for large $n$): for normal data the mean is the more precise
estimator. The sample SD is centred at 0.591, slightly *below* $\sigma = 0.6$ — even with the $n - 1$ divisor, $s$ is
a little biased (only $s^2$ is unbiased) — and its histogram is slightly skewed.

### 6.2 The standard error of an allele frequency

A sample proportion is a mean of 0/1 values, so everything above applies. An allele frequency estimated from $n$ diploid
people is the mean of $2n$ allele indicators with $\sigma^2 = p(1-p)$, hence $\operatorname{SE}(\hat p) = \sqrt{p(1-p)/(2n)}$
(if the two alleles of each person are independent — Hardy–Weinberg). Module 23 will show that this $\hat p$ is also the
maximum-likelihood estimate.

In [ ]:
# ── 6.2 Allele-frequency estimates from SIMULATED Hardy–Weinberg genotypes: SE vs √(p(1 − p)/(2n)) ──
p_true = 0.3
rows = []
for n_people_ in [50, 200, 1000]:
    p_hats = np.array([simulate_genotypes(n_people_, p_true, seed=s).sum() / (2 * n_people_) for s in range(2000)])
    rows.append({"people n": n_people_, "mean of p̂": p_hats.mean(), "SD of p̂ (2,000 studies)": p_hats.std(),
                 "√(p(1−p)/(2n))": sqrt(p_true * (1 - p_true) / (2 * n_people_)),
                 "95 % of p̂ within": f"[{np.quantile(p_hats, 0.025):.3f}, {np.quantile(p_hats, 0.975):.3f}]"})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

🔍 **What just happened?** For 2,000 simulated studies at each size, the estimates are centred on the true 0.3 (unbiased),
and their spread matches the formula: 0.0460 (formula 0.0458) for 50 people, 0.0105 (formula 0.0102) for 1,000. With 50 people the middle 95 % of
estimates span roughly 0.21 to 0.39 — an allele frequency from 50 people is only good to about ±0.09. This is why genetic
association studies need thousands of participants.

✏️ **Your turn.** (a) Replace the normal data of 6.1 by our bursty mRNA counts (`simulate_mrna_per_cell`). Which has the
smaller SE now, the mean or the median — and what do they estimate (are they centred at the same value)? (b) Using the
formula, how many people do you need for $\operatorname{SE}(\hat p) = 0.005$ when $p = 0.3$?

---
## 7 · Confidence intervals for a mean

> 📍 **Where we are.** A point statistic like $\bar X_n$ says nothing about its own precision. An *interval statistic*
> does: the CLT tells us how far $\bar X_n$ typically strays from $\mu$, so we can turn that around and build an interval
> around $\bar X_n$ that catches $\mu$ most of the time.

**Known $\sigma$: the z interval.** By the CLT, $Z_n = (\bar X_n - \mu)/(\sigma/\sqrt n) \approx \mathcal N(0, 1)$. Let
$z = 1.96$, the value with $P(-z \le Z \le z) = 0.95$ (`stats.norm.ppf(0.975)`). Rearrange the event inside the
probability, one step at a time:

$$\begin{aligned}
0.95 &\approx P\Big(-z \le \frac{\bar X_n - \mu}{\sigma/\sqrt n} \le z\Big) \\
&= P\Big(-z\frac{\sigma}{\sqrt n} \le \bar X_n - \mu \le z\frac{\sigma}{\sqrt n}\Big) &&\text{multiply by } \sigma/\sqrt n > 0\\
&= P\Big(-\bar X_n - z\frac{\sigma}{\sqrt n} \le -\mu \le -\bar X_n + z\frac{\sigma}{\sqrt n}\Big) &&\text{subtract } \bar X_n\\
&= P\Big(\bar X_n - z\frac{\sigma}{\sqrt n} \le \mu \le \bar X_n + z\frac{\sigma}{\sqrt n}\Big) &&\text{multiply by } -1 \text{ (flips the inequalities)}
\end{aligned}$$

*In words:* the random interval $\bar X_n \pm 1.96\,\sigma/\sqrt n$ contains the fixed number $\mu$ with probability about
0.95. That interval is the **95 % confidence interval** (CI) for $\mu$; $1.96\,\sigma/\sqrt n$ is its **margin of error**.
For a general confidence level $1 - \alpha$, replace 1.96 by $z_{\alpha/2} = \Phi^{-1}(1 - \alpha/2)$ (1.645 for 90 %,
2.576 for 99 %).

🧮 **Math refresher — quantiles.** $\Phi^{-1}(q)$ (`stats.norm.ppf(q)`) is the value below which a standard normal falls
with probability $q$: $\Phi^{-1}(0.975) = 1.96$ means $P(Z \le 1.96) = 0.975$, so 2.5 % lies above 1.96 and, by
symmetry, 2.5 % below −1.96.

### 7.1 One hundred experiments, one hundred intervals

In [ ]:
# ── 7.1 100 z intervals for the mean mRNA count, n = 30 cells each, σ known (SIMULATED) ──
def z_interval(x, sigma, conf=0.95):
    """CLT confidence interval x̄ ± z σ/√n for the mean, with the population SD σ known."""
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    half = z * sigma / np.sqrt(len(x))
    return np.mean(x) - half, np.mean(x) + half


rng_ci = np.random.default_rng(SEED + 71)
intervals = np.array([z_interval(simulate_mrna_per_cell(30, rng_ci), TRUE_SD) for _ in range(100)])
covers = (intervals[:, 0] <= TRUE_MEAN) & (TRUE_MEAN <= intervals[:, 1])
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for i, (lo, hi) in enumerate(intervals):
    ax.plot([lo, hi], [i, i], color="tab:blue" if covers[i] else "tab:red", lw=1.4 if not covers[i] else 1)
    ax.plot((lo + hi) / 2, i, ".", color="k", ms=2)
ax.axvline(TRUE_MEAN, color="k", ls="--", lw=1, label="true mean μ = 6")
ax.set_xlabel("mRNA molecules per cell"); ax.set_ylabel("experiment")
ax.set_title(f"SIMULATED: 100 experiments × 30 cells — {covers.sum()} intervals catch μ, {100 - covers.sum()} miss (red)")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()
print(f"margin of error = 1.96 × {TRUE_SD:.3f}/√30 = {1.96 * TRUE_SD / sqrt(30):.3f} molecules")

🔍 **What just happened?** Each horizontal line is the 95 % interval from one simulated experiment of 30 cells; all have
the same width ($\pm 1.96 \times 5.477/\sqrt{30} = \pm 1.96$ molecules) but different centres. 94 of the 100
intervals contain the true mean 6 (blue); 6 miss it (red). We expected about 95 and 5. In a real experiment you see
*one* of these lines and never know whether yours is blue or red — the 95 % describes the *method*.

In [ ]:
# ── 7.2 Long-run coverage of the z interval for skewed data: 10,000 experiments per sample size ──
rng_cov = np.random.default_rng(SEED + 72)
for n_ in [5, 30, 100]:
    x = simulate_mrna_per_cell(10_000 * n_, rng_cov).reshape(10_000, n_)
    half = 1.96 * TRUE_SD / sqrt(n_)
    xbar = x.mean(axis=1)
    cover = np.mean(np.abs(xbar - TRUE_MEAN) <= half)
    miss_high = np.mean(xbar - half > TRUE_MEAN)            # whole interval above μ
    miss_low = np.mean(xbar + half < TRUE_MEAN)             # whole interval below μ
    print(f"n = {n_:3d} cells: coverage of x̄ ± 1.96 σ/√n = {cover:.4f}  (nominal 0.95);  "
          f"misses above μ: {miss_high:.4f}, below μ: {miss_low:.4f}  (0.025 each if symmetric)")

🔍 **What just happened?** Over 10,000 experiments the z interval covers the true mean 0.958 of the time with 5 cells,
0.952 with 30 and 0.953 with 100 — all close to the nominal 0.95. But look at *where* the misses happen: with 5 cells,
3.9 % of intervals lie entirely above $\mu$ and only 0.3 % entirely below, instead of 2.5 % each. Right-skewed counts make
$\bar X_n$ right-skewed (Section 4), so the interval errs on the high side; the imbalance shrinks as $n$ grows (2.8 % vs
2.0 % at $n = 100$). The 95 % is an approximation that rests on the CLT — total coverage can look fine while the two
tails are wrong, and for other skewed distributions or tiny samples coverage can also fall short.

### 7.3 What a 95 % confidence interval does — and does not — mean

* ✅ **It does mean:** if the experiment were repeated many times and an interval computed each time, about 95 % of those
  intervals would contain $\mu$ (Section 7.1). The probability belongs to the *procedure*, before the data are seen.
* ❌ It does **not** mean "there is a 95 % probability that $\mu$ lies in *this* interval [5.1, 9.0]". In the frequentist
  view $\mu$ is a fixed number: a given interval either contains it or not. (A Bayesian *credible* interval does carry
  that interpretation — Module 24.)
* ❌ It does **not** mean that 95 % of the *cells* (data values) lie in the interval. The CI is about the mean: its
  half-width is $1.96\,\sigma/\sqrt n$, while single cells scatter with SD $\sigma$ — $\sqrt n$ times more widely.
* ❌ It does **not** mean that a replicate experiment's mean has a 95 % chance of falling in this interval (that
  probability is lower — about 83 % — because both means wobble).

### 7.4 Unknown $\sigma$: the t interval (preview)

In practice $\sigma$ is unknown and we plug in the sample SD $s$. That adds extra uncertainty — $s$ itself wobbles
(Section 6.1) — so for **normal data** the standardised mean follows a slightly wider distribution, Student's $t$ with
$n - 1$ degrees of freedom (Module 25 studies it):

$$T = \frac{\bar X_n - \mu}{s/\sqrt n} \sim t_{n-1}, \qquad\text{95\% CI: } \bar X_n \pm t_{n-1,\,0.975}\,\frac{s}{\sqrt n} .$$

*In words:* the same recipe as the z interval, with $\sigma$ replaced by $s$ and 1.96 replaced by a larger number that
depends on how many observations estimated $s$.

In [ ]:
# ── 7.4 t quantiles, and why they matter for small n (normal data, SIMULATED log2 expression) ──
print("n        :", "  ".join(f"{n_:6d}" for n_ in [3, 5, 10, 30, 100]), "     ∞")
print("t_{n−1}  :", "  ".join(f"{stats.t.ppf(0.975, n_ - 1):6.3f}" for n_ in [3, 5, 10, 30, 100]),
      f"{stats.norm.ppf(0.975):6.3f}")


def t_interval(x, conf=0.95):
    """Student-t confidence interval x̄ ± t_{n−1} s/√n for the mean of (approximately normal) data."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    half = stats.t.ppf(1 - (1 - conf) / 2, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return x.mean() - half, x.mean() + half


rng_t = np.random.default_rng(SEED + 74)
x5 = rng_t.normal(mu_ex, sigma_ex, (20_000, 5))                  # 20,000 experiments with 5 mice each
m5, s5 = x5.mean(axis=1), x5.std(axis=1, ddof=1)
cover_z_with_s = np.mean(np.abs(m5 - mu_ex) <= 1.96 * s5 / sqrt(5))
cover_t = np.mean(np.abs(m5 - mu_ex) <= stats.t.ppf(0.975, 4) * s5 / sqrt(5))
print(f"\nn = 5 mice: coverage of x̄ ± 1.96 s/√n = {cover_z_with_s:.3f};  coverage of x̄ ± t_4 s/√n = {cover_t:.3f}")
lo, hi = t_interval(x5[0])
print(f"first experiment: x̄ = {m5[0]:.3f}, s = {s5[0]:.3f}, 95 % t interval = [{lo:.3f}, {hi:.3f}]  (μ = {mu_ex})")
print("check with scipy:", np.round(stats.t.interval(0.95, 4, loc=m5[0], scale=s5[0] / sqrt(5)), 3))

🔍 **What just happened?** The $t$ multipliers are 4.303 for $n = 3$, 2.776 for $n = 5$, 2.262 for $n = 10$, 2.045 for
$n = 30$ and approach 1.960 as $n \to \infty$. With 5 mice, naively using 1.96 with the sample SD covers $\mu$ only
0.876 of the time — far below 95 % — while the $t$ interval covers 0.951, as promised. Our hand-written `t_interval`
agrees with `stats.t.interval`. Rule: **with an estimated SD and a small sample, use $t$.**

### 7.5 Biology: the cholesterol drug trial

The toolkit's trial measured LDL cholesterol (mg/dL) in 12 patients before and after treatment. Each patient is their
own control, so we analyse the 12 **differences** $d_i = \text{after}_i - \text{before}_i$ and build a t interval for the
mean change $\mu_d$ (the *paired* analysis that Module 25 turns into a test).

In [ ]:
# ── 7.5 95 % t interval for the mean LDL change (SIMULATED trial, hidden true effect −18 mg/dL) ──
trial = simulate_cholesterol_trial()
d = (trial["after"] - trial["before"]).to_numpy()
print(trial.assign(change=d.round(1)).to_string(index=False))
lo, hi = t_interval(d)
print(f"\nn = {len(d)}, mean change d̄ = {d.mean():.2f} mg/dL, s_d = {d.std(ddof=1):.2f}, "
      f"SE = s_d/√n = {d.std(ddof=1) / sqrt(len(d)):.2f}, t_11 = {stats.t.ppf(0.975, 11):.3f}")
print(f"95 % CI for the mean change: [{lo:.1f}, {hi:.1f}] mg/dL   🧪 (true effect in the simulator: −18)")

🔍 **What just happened?** The 12 patients' LDL changed by −12.88 mg/dL on average — a fall of nearly 13 (SD of the changes 12.79, SE 3.69).
The 95 % t interval [−21.0, −4.7] lies entirely below 0 — the data are incompatible with "no change" at the 95 % level
(Module 25 makes this a paired t-test) — and it contains the simulator's hidden true effect of −18 mg/dL. Notice that the SD of the
changes (12.8) is far smaller than the patients' spread of LDL levels (SD about 30 mg/dL): pairing removes the
between-patient differences, which is why 12 patients are enough to see a change of this size.

✏️ **Your turn.** (a) Compute a 99 % interval for the LDL change. Is it wider or narrower, and does it still exclude 0?
(b) By hand: 16 mice give $\bar x = 7.9$ and $s = 0.8$; $t_{15, 0.975} = 2.131$. Write the 95 % CI.

### 📘 Lecture notes — confidence intervals

**Interval statistic.** A confidence interval $[L, U]$ is computed from the data, so it is random; a $(1-\alpha)$ CI
satisfies $P(L \le \mu \le U) \approx 1 - \alpha$ *before* the data are seen.

**z interval (σ known, CLT).** From $P(-z_{\alpha/2} \le \frac{\bar X_n - \mu}{\sigma/\sqrt n} \le z_{\alpha/2}) \approx 1 - \alpha$, rearranged:

$$\bar X_n \pm z_{\alpha/2}\frac{\sigma}{\sqrt n}, \qquad z_{0.05} = 1.645,\ z_{0.025} = 1.960,\ z_{0.005} = 2.576 .$$

**t interval (σ unknown, normal data).** $\bar X_n \pm t_{n-1,\,1-\alpha/2}\,s/\sqrt n$ with
$t_{4} = 2.776$, $t_{11} = 2.201$, $t_{29} = 2.045$ for 95 %; $t \to z$ as $n \to \infty$. Using 1.96 with $s$ and $n = 5$
covers only 0.876.

**Interpretation.** 95 % of intervals from repeated experiments contain $\mu$ (simulation: 94 of 100). It is **not** the
probability that $\mu$ is in one given interval, not the range of 95 % of the data, and not the range of a replicate's mean.

**Worked example.** Cholesterol trial (12 patients, paired differences): $\bar d =$ −12.88, $s_d =$ 12.79, 95 % CI [−21.0, −4.7] mg/dL.

> 🎓 **Exam tip.** Write the CI derivation as a chain of equivalent events (multiply by $\sigma/\sqrt n$, subtract
> $\bar X_n$, multiply by −1 and flip). And always say which multiplier you used — $z$ or $t_{n-1}$ — and why.

---
## 8 · Planning a study: sample size and replicates

> 📍 **Where we are.** The margin of error $z\sigma/\sqrt n$ contains $n$ — so we can choose $n$ *before* the experiment
> to reach a desired precision. This is the most practical use of the CLT in experimental biology.

**Sample size for a margin of error $E$.** Set the half-width equal to $E$ and solve for $n$:

$$E = z\frac{\sigma}{\sqrt n} \;\Longrightarrow\; \sqrt n = \frac{z\sigma}{E} \;\Longrightarrow\; n = \Big(\frac{z\,\sigma}{E}\Big)^2
\quad\text{(round up)} .$$

*In words:* the required sample size grows with the square of the noise $\sigma$ and with the square of the precision
you demand ($1/E$). Halving the margin of error quadruples the sample. $\sigma$ must come from a pilot study or the
literature — an honest guess, labelled as such.

In [ ]:
# ── 8.1 How many patients, how many mice? (illustrative σ values) ──
def sample_size_for_margin(sigma, margin, conf=0.95):
    """Smallest n with z σ/√n ≤ margin (CLT, σ treated as known)."""
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    return int(np.ceil((z * sigma / margin) ** 2))


sigma_d = sqrt(10 ** 2 + 10 ** 2)                 # SD of a paired LDL difference if day-to-day noise has SD 10
plans = [("LDL change, margin ±5 mg/dL", sigma_d, 5.0), ("LDL change, margin ±2.5 mg/dL", sigma_d, 2.5),
         ("mouse log2 expression (σ = 0.5), margin ±0.25", 0.5, 0.25),
         ("mouse log2 expression (σ = 0.5), margin ±0.1", 0.5, 0.1),
         ("mRNA per cell (σ = 5.48), margin ±0.5 molecules", TRUE_SD, 0.5)]
for name, s, e in plans:
    print(f"{name:48s} σ = {s:5.2f} → n = {sample_size_for_margin(s, e):5d}")

🔍 **What just happened?** If the day-to-day noise of an LDL measurement has SD 10 mg/dL, a before–after difference has
SD $\sqrt{10^2 + 10^2} = 14.1$, and a ±5 mg/dL margin needs $n = (1.96 \times 14.14/5)^2 = 30.7 \to 31$ patients; halving
the margin to ±2.5 needs 123 (four times as many). For mouse expression with $\sigma = 0.5$ log2 units, ±0.25 needs 16
mice and ±0.1 needs 97. For single cells with our bursty gene, ±0.5 molecules needs 461 cells — cheap with imaging,
expensive with mice. (Planning for a *test*, with power, comes in Module 25.)

### 8.2 Biological vs technical replicates

A **biological replicate** is a new, independent biological unit (another mouse, another patient, another culture); a
**technical replicate** is a repeated measurement of the *same* unit (the same RNA run twice). Model a measurement of
mouse $i$, replicate $j$ as $Y_{ij} = \mu + B_i + T_{ij}$, where $B_i$ (variance $\sigma_B^2$) is the mouse's own deviation
and $T_{ij}$ (variance $\sigma_T^2$) is measurement noise, all independent. With $m$ mice and $t$ technical replicates
each, the grand mean is

$$\bar Y = \mu + \frac1m\sum_{i=1}^m B_i + \frac{1}{mt}\sum_{i=1}^m\sum_{j=1}^t T_{ij}, \qquad
\operatorname{Var}(\bar Y) = \frac{m\,\sigma_B^2}{m^2} + \frac{mt\,\sigma_T^2}{(mt)^2} = \frac{\sigma_B^2}{m} + \frac{\sigma_T^2}{mt} .$$

*In words:* technical replicates only shrink the second term. However many times you re-measure the same mice, the
error can never fall below $\sigma_B/\sqrt m$ — the biological variation is averaged only over *independent* animals.

In [ ]:
# ── 8.2 Same budget of 24 measurements, five designs (SIMULATED: σ_B = 0.6, σ_T = 0.25 log2 units) ──
def simulate_replicate_design(n_mice, n_tech, rng, mu=8.0, sd_bio=0.6, sd_tech=0.25):
    """SIMULATED log2 expression: n_mice independent mice (biological variation sd_bio), each measured n_tech times
    (technical noise sd_tech). Returns an (n_mice, n_tech) array."""
    mouse_level = mu + sd_bio * rng.standard_normal(n_mice)
    return mouse_level[:, None] + sd_tech * rng.standard_normal((n_mice, n_tech))


rng_rep = np.random.default_rng(SEED + 82)
rows = []
for m, t in [(24, 1), (12, 2), (6, 4), (3, 8), (1, 24)]:
    grand_means = [simulate_replicate_design(m, t, rng_rep).mean() for _ in range(20_000)]
    rows.append({"mice m": m, "technical t": t, "SE by formula": sqrt(0.6 ** 2 / m + 0.25 ** 2 / (m * t)),
                 "SE by simulation": np.std(grand_means), "floor σ_B/√m": 0.6 / sqrt(m)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

🔍 **What just happened?** All five designs cost 24 measurements, but their standard errors differ by a factor of about 4.5:
24 mice measured once give SE 0.1328 (formula 0.1327), while one mouse measured 24 times gives 0.6057 (formula 0.6022) —
no better than measuring that mouse once, because the dominant error is the mouse-to-mouse variation. Simulation and
formula agree in every row, and every design sits just above its floor $\sigma_B/\sqrt m$.

### 🌉 Bridge — replication in genomics: count the right $n$

The $n$ in $\sigma/\sqrt n$ is the number of **independent biological units**. RNA-seq experiments with three technical
replicates of one pooled sample have $n = 1$ for any statement about the population; cells from one mouse are not
independent replicates of a treatment (they share the mouse), which is why single-cell differential-expression analyses
now "pseudobulk" cells per animal before testing. Blainey, Krzywinski & Altman (2014) give a short, clear account of
replication for biologists. The variance-components model of Section 8.2 is the simplest **mixed model**, the workhorse of
designs with nested replication; for batch effects across experiments see
[BiologyToML 31 · batch integration](../../../../ml/tutorial/BiologyToML/notebooks/31_batch_integration.ipynb), and for single-cell data
[BiologyToML 30](../../../../ml/tutorial/BiologyToML/notebooks/30_single_cell_rnaseq.ipynb).

✏️ **Your turn.** With $\sigma_B = 0.6$ and $\sigma_T = 0.25$, a mouse costs 10 units and a measurement 1 unit. With a
budget of 100 units, which $(m, t)$ minimises $\operatorname{Var}(\bar Y)$? (Write $\operatorname{Var}$ as a function of $m$ with
$t = (100 - 10m)/m$ measurements per mouse and evaluate it for $m = 1, \dots, 9$.)

---
## 9 · The bootstrap: an interval when there is no formula

> 📍 **Where we are.** Sections 6–7 relied on formulas for the standard error. For many statistics — a median, a ratio,
> a correlation, a tree — there is no simple formula. If we could rerun the experiment thousands of times we would just
> look at the spread, as in Section 6. The bootstrap pretends we can.

**Idea (Efron, 1979).** The data $x_1, \dots, x_n$ are our best picture of the unknown distribution $P(X)$. So draw a new
"data set" of size $n$ **with replacement** from $x_1, \dots, x_n$ (some values appear twice, some not at all), compute the
statistic on it, and repeat $B$ times (say $B = 5000$). The spread of these $B$ **bootstrap replicates** approximates the
sampling distribution:

* bootstrap standard error = SD of the $B$ replicates;
* 95 % **percentile interval** = the 2.5 % and 97.5 % quantiles of the replicates.

*In words:* resampling the data imitates resampling the population, as long as the data are i.i.d. and $n$ is not tiny.

In [ ]:
# ── 9.1 Bootstrap interval for the MEDIAN mRNA count of 40 cells (SIMULATED) — and a check against the truth ──
def bootstrap_replicates(x, statistic, n_boot, rng):
    """Values of `statistic` on n_boot resamples (with replacement, same size) of the data x."""
    x = np.asarray(x)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    return np.array([statistic(x[row]) for row in idx])


rng_b = np.random.default_rng(SEED + 91)
sample40 = simulate_mrna_per_cell(40, rng_b)
boot_med = bootstrap_replicates(sample40, np.median, 5000, rng_b)
ci_boot = np.quantile(boot_med, [0.025, 0.975])
true_median = stats.nbinom.median(BURST_SHAPE, BURST_SHAPE / (BURST_SHAPE + TRUE_MEAN))
truth_meds = np.median(simulate_mrna_per_cell(40 * 20_000, rng_b).reshape(20_000, 40), axis=1)
print(f"sample median of 40 cells = {np.median(sample40):.1f}; true median of the tissue = {true_median:.0f}")
print(f"bootstrap SE of the median = {boot_med.std():.2f}   (true SE from 20,000 new experiments = {truth_meds.std():.2f})")
print(f"95 % bootstrap percentile interval for the median: [{ci_boot[0]:.1f}, {ci_boot[1]:.1f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(boot_med, bins=np.arange(0, 12, 0.5), color="tab:orange", alpha=0.7, density=True)
axes[0].axvline(np.median(sample40), color="k", ls="--", label="median of our 40 cells")
axes[0].set_title("bootstrap: 5,000 resamples of OUR 40 cells"); axes[0].set_xlabel("median"); axes[0].legend()
axes[1].hist(truth_meds, bins=np.arange(0, 12, 0.5), color="tab:gray", alpha=0.7, density=True)
axes[1].axvline(true_median, color="tab:red", ls="--", label="true median")
axes[1].set_title("truth: 20,000 NEW experiments of 40 cells"); axes[1].set_xlabel("median"); axes[1].legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** From one sample of 40 cells (median 5) the bootstrap estimates the standard error of the
median as 0.94; 20,000 genuinely new experiments — possible only because we simulate — give a true SE of 0.95. The
two histograms have similar spreads. Here our sample's median happens to equal the true median (5), but in general the
bootstrap histogram is centred on *our* sample's value, not on the truth: the bootstrap estimates the **spread**, not the
location. The 95 % percentile interval [4.0, 7.5] contains the true median
5. Medians of counts are lumpy (only integers and half-integers occur), which is why the histograms look jagged.

In [ ]:
# ── 9.2 Bootstrap vs t interval for the mean LDL change of the cholesterol trial ──
rng_b2 = np.random.default_rng(SEED + 92)
boot_mean_d = bootstrap_replicates(d, np.mean, 5000, rng_b2)
print(f"t interval:                  [{t_interval(d)[0]:.1f}, {t_interval(d)[1]:.1f}] mg/dL")
print(f"bootstrap percentile (5000): [{np.quantile(boot_mean_d, 0.025):.1f}, {np.quantile(boot_mean_d, 0.975):.1f}] mg/dL")
print(f"bootstrap SE = {boot_mean_d.std():.2f} vs s/√n = {d.std(ddof=1) / sqrt(len(d)):.2f}")

🔍 **What just happened?** For the mean of the 12 LDL changes, the bootstrap percentile interval [−19.3, −5.9] is close to the
t interval [−21.0, −4.7] but slightly narrower. That is typical for small $n$: the bootstrap SE 3.44 divides by $n$ rather
than $n - 1$ and uses normal-like quantiles, so it under-covers a little; with $n = 12$ the $t$ interval is preferable
for a mean of roughly normal data. The bootstrap earns its keep for statistics without formulas.

### 🌉 Bridge — resampling everywhere: phylogenies, machine learning and conformal prediction

The bootstrap is one of the most used ideas in computational biology. **Phylogenetics:** Felsenstein (1985) resampled
alignment *columns* to put "bootstrap support" values on the branches of evolutionary trees — the numbers you see on
almost every published tree. **Machine learning:** bagging and random forests train models on bootstrap resamples
([BiologyToML 10](../../../../ml/tutorial/BiologyToML/notebooks/10_decision_trees_random_forests.ipynb)); cross-validation, a cousin of the bootstrap, estimates
prediction error ([BiologyToML 23 · model selection](../../../../ml/tutorial/BiologyToML/notebooks/23_model_selection.ipynb)); and **conformal prediction** builds
prediction intervals with guaranteed coverage from held-out residuals — the same "95 % of the time" logic as Section 7,
for predictions instead of means ([BiologyToML 42 · trustworthy ML](../../../../ml/tutorial/BiologyToML/notebooks/42_trustworthy_ml.ipynb)).

✏️ **Your turn.** (a) Compute a bootstrap 95 % interval for the *standard deviation* of the 40 cells. Does it contain the
true $\sigma = 5.48$? (b) Resample the 40 cells but compute the *fraction of cells with zero molecules*. Compare its
bootstrap SE with the formula $\sqrt{\hat p(1 - \hat p)/n}$.

### 🌉 Bridge — averaging noisy images in cryo-electron microscopy

A single cryo-EM image of a protein is so noisy that the molecule is barely visible: the signal-to-noise ratio is far
below 1. Averaging $N$ aligned images of identical particles keeps the signal and shrinks the noise SD by $\sqrt N$ —
exactly $\operatorname{SD}(\bar X_N) = \sigma/\sqrt N$, applied pixel by pixel. Averaging 10,000 particles improves the
SNR 100-fold, which is how near-atomic structures emerge from images that individually look like static. The same
square-root law limits how fast resolution improves with more data. See
[Cryo-EM · noise, statistics and inference](../../../../cryoem/tutorial/BiologyToResearch/notebooks/04_noise_statistics_inference.ipynb).

In [ ]:
# ── 9.3 The √N law in pixels: averaging SIMULATED noisy images of a ring-shaped "particle" ──
yy, xx = np.mgrid[-1:1:64j, -1:1:64j]
particle = np.exp(-((np.sqrt(xx ** 2 + yy ** 2) - 0.5) ** 2) / 0.01)          # a ring, signal amplitude 1
rng_img = np.random.default_rng(SEED + 93)
noise_sd = 3.0                                                                # SNR (amplitude/noise SD) = 1/3
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, n_img in zip(axes, [1, 10, 100, 1000]):
    avg = particle + noise_sd * rng_img.standard_normal((n_img, 64, 64)).mean(axis=0)
    resid_sd = (avg - particle).std()
    ax.imshow(avg, cmap="gray"); ax.axis("off")
    ax.set_title(f"N = {n_img}: noise SD {resid_sd:.2f}\n(σ/√N = {noise_sd / np.sqrt(n_img):.2f})", fontsize=8)
fig.suptitle("SIMULATED: averaging N aligned noisy images of the same particle", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** One image (noise SD 3, three times the signal) shows nothing. Averaging 10 images leaves noise
SD 0.94, 100 images 0.30, 1,000 images 0.10 — matching $3/\sqrt N$ — and the ring emerges. Each pixel's
average obeys the same law as the mean of 30 cells.

## 📌 Key takeaways

- **Statistics** makes inferences from data in three phases — collect (after forming hypotheses), describe, analyse. A
  **statistic** is anything computable from the data; it is a **random variable** because new data give a new value.
- For i.i.d. data, the sample mean $\bar X_n$ is a random variable with $E[\bar X_n] = \mu$ and
  $\operatorname{Var}(\bar X_n) = \sigma^2/n$; its standard error $\sigma/\sqrt n$ halves only when $n$ quadruples.
- **Law of large numbers:** $P(|\bar X_n - \mu| < \varepsilon) \to 1$ for every $\varepsilon > 0$ (not
  $P(\bar X_n = \mu) \to 1$). Chebyshev proves it from $\operatorname{Var}(\bar X_n) = \sigma^2/n$; it fails without a
  finite mean (Cauchy).
- **Central limit theorem:** $(\bar X_n - \mu)/(\sigma/\sqrt n) \to \mathcal N(0, 1)$, i.e. $\bar X_n \approx \mathcal N(\mu, \sigma^2/n)$
  and $S_n \approx \mathcal N(n\mu, n\sigma^2)$ — for almost any data distribution, sooner for symmetric data, later for skewed
  or rare-event data. It explains bell-shaped polygenic traits.
- The CLT gives **approximate probabilities** when the exact distribution is unknown or expensive, and tells us whether
  a **sample size is adequate** ($n = 385$ flips for ±0.05 with 95 % probability; Chebyshev would demand 2000).
- A **95 % confidence interval** $\bar X_n \pm 1.96\,\sigma/\sqrt n$ (or $\pm t_{n-1}\,s/\sqrt n$ when $\sigma$ is estimated) catches
  $\mu$ in 95 % of repeated experiments; it is not a 95 % probability statement about one interval.
- **Planning:** $n = (z\sigma/E)^2$; only **biological** replicates reduce the biological part of the error
  ($\operatorname{Var}(\bar Y) = \sigma_B^2/m + \sigma_T^2/(mt)$).
- The **bootstrap** approximates a sampling distribution by resampling the data — error bars for medians, trees and
  anything else without a formula.

---
## ✅ Checkpoint

1. A gene's expression varies between mice with $\sigma = 0.8$ log2 units. What is the standard error of the mean of 4
   mice? Of 16 mice?
   <details><summary>Answer</summary>$0.8/\sqrt4 = 0.4$ and $0.8/\sqrt{16} = 0.2$: four times the mice, half the error.</details>
2. Derive $\operatorname{Var}(\bar X_n) = \sigma^2/n$ and say exactly where independence is used.
   <details><summary>Answer</summary>$\operatorname{Var}(\frac1n\sum X_i) = \frac1{n^2}\operatorname{Var}(\sum X_i)$ (constants come out squared) $= \frac1{n^2}\sum\operatorname{Var}(X_i)$ (**independence**: no covariance terms) $= \frac{n\sigma^2}{n^2} = \sigma^2/n$. $E[\bar X_n] = \mu$ needs no independence.</details>
3. 30 cells come as 15 pairs of sister cells with identical counts (each pair independent of the others). What is $\operatorname{Var}(\bar X_{30})$?
   <details><summary>Answer</summary>$\bar X_{30}$ is the mean of 15 independent pair values (each counted twice), so $\operatorname{Var} = \sigma^2/15$ — twice the $\sigma^2/30$ of independent cells. Correlated data contain less information than their count suggests.</details>
4. Why is "$\lim_{n\to\infty} P(\bar X_n = \mu) = 1$" a wrong statement of the law of large numbers? Give the correct one.
   <details><summary>Answer</summary>For continuous data $P(\bar X_n = \mu) = 0$ for every $n$; even for a fair coin $P(\bar X_{n} = 1/2) \to 0$ (0.0080 at $n = 10\,000$). Correct: $\lim_{n\to\infty} P(|\bar X_n - \mu| < \varepsilon) = 1$ for every $\varepsilon > 0$.</details>
5. Use Chebyshev to find $n$ guaranteeing $P(|\bar X_n - 6| \ge 1) \le 0.05$ for mRNA counts with $\sigma^2 = 30$. What does the CLT suggest?
   <details><summary>Answer</summary>Chebyshev: $30/(n \cdot 1^2) \le 0.05 \Rightarrow n \ge 600$. CLT: $1.96\sqrt{30}/\sqrt n \le 1 \Rightarrow n \ge 115.2$, so 116 — five times fewer cells.</details>
6. Data have $\mu = 4.2$, $\sigma^2 = 11.76$ per cell. Give the approximate distribution of the total $S_{50}$ and of the mean $\bar X_{50}$ (with variances).
   <details><summary>Answer</summary>$S_{50} \approx \mathcal N(210,\ 588)$ (SD 24.25); $\bar X_{50} \approx \mathcal N(4.2,\ 0.2352)$ (SD 0.485).</details>
7. Approximate $P(S_{100} \ge 60)$ for 100 fair flips with and without continuity correction.
   <details><summary>Answer</summary>$\mu = 50$, $\sigma_S = 5$. Without: $z = 2$, $1 - \Phi(2) = 0.0228$. With: $z = (59.5 - 50)/5 = 1.9$, $0.0287$. Exact: 0.0284.</details>
8. 16 mice give $\bar x = 7.9$, $s = 0.8$; $t_{15,0.975} = 2.131$. Give the 95 % CI. Why not use 1.96?
   <details><summary>Answer</summary>$7.9 \pm 2.131 \times 0.8/4 = 7.9 \pm 0.426 = [7.474, 8.326]$. $\sigma$ is estimated by $s$, which adds uncertainty; the $t_{15}$ multiplier accounts for it (1.96 would under-cover).</details>
9. A paper reports "95 % CI for the mean LDL change: [−24, −12] mg/dL" and concludes "there is a 95 % probability that the true change is between −24 and −12". What is wrong?
   <details><summary>Answer</summary>In the frequentist framework the true change is fixed; this particular interval either contains it or not. The 95 % refers to the procedure: 95 % of intervals built this way in repeated trials contain the truth. (A Bayesian credible interval, Module 24, supports the probability statement.)</details>
10. How many patients give a ±5 mg/dL margin (95 %) for a mean LDL change with SD 14.1? With ±2.5?
    <details><summary>Answer</summary>$n = (1.96 \times 14.1/5)^2 = 30.6 \to 31$; for ±2.5, four times as many: $122.2 \to 123$ (with $\sigma = 14.14$: 31 and 123).</details>
11. With $\sigma_B = 0.6$ and $\sigma_T = 0.25$, compare the SE of the grand mean for 12 mice × 2 technical replicates and 3 mice × 8.
    <details><summary>Answer</summary>$\sqrt{0.36/12 + 0.0625/24} = \sqrt{0.0300 + 0.0026} = 0.181$; $\sqrt{0.36/3 + 0.0625/24} = \sqrt{0.1200 + 0.0026} = 0.350$. Biological replicates win.</details>
12. In one sentence each: what does the bootstrap estimate well, and what can it not fix?
    <details><summary>Answer</summary>It estimates the *spread* (SE, interval width) of a statistic's sampling distribution from one data set; it cannot fix a biased or unrepresentative sample — it is centred on the sample's own value, and it assumes the data are i.i.d.</details>

## 🧠 Questions & discussion

1. The LLN and the CLT assume i.i.d. data. Give two biological sampling schemes where independence fails (cells from the
   same clone, patients from the same family, time series), and discuss what happens to $\operatorname{Var}(\bar X_n)$.
2. Height is approximately normal, but body weight and many expression levels are right-skewed. Using the CLT, suggest
   why a *multiplicative* combination of many small effects produces a log-normal rather than a normal distribution.
3. "We used $n = 3$ because that is standard in the field." Using Sections 7–8, argue for or against this practice for
   a mouse experiment with a between-animal SD of 0.6 log2 units.
4. A 95 % CI excludes 0. A colleague says "so the effect is real with 95 % certainty". Discuss what the CI does and does
   not justify, and what else (design, multiple comparisons — Module 26) matters.
5. When is a *median* a better summary of single-cell counts than a mean? What would you report, with which error bar?

## 🚀 Bonus — going further

- **Berry–Esseen.** The error of the CLT approximation is at most $C\,\rho/(\sigma^3\sqrt n)$ with $\rho = E|X - \mu|^3$ and a
  constant $C < 0.48$. Compute the bound for Bernoulli(0.1) and $n = 30$, and compare with the gap you saw in Section 4.1.
- **The delta method.** If $\bar X_n \approx \mathcal N(\mu, \sigma^2/n)$, then $g(\bar X_n) \approx \mathcal N(g(\mu), g'(\mu)^2\sigma^2/n)$. Use it to
  get a standard error for $\log \bar X_n$ (a log fold change) and check it by simulation.
- **Wilson interval for a proportion.** The interval $\hat p \pm 1.96\sqrt{\hat p(1 - \hat p)/n}$ fails badly for small $n$ or
  extreme $p$ (try $n = 20$, $p = 0.05$ by simulation). Look up the Wilson score interval and compare coverages.
- **Bootstrap variants.** Compare the percentile interval with the *basic* and *BCa* intervals in
  `scipy.stats.bootstrap` for the median of skewed counts.
- **Stable laws.** Why does the mean of Cauchy variables stay Cauchy? Read about stable distributions — the "CLT" for
  infinite-variance data.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            22 · Sampling: the law of large numbers, the CLT and confidence intervals
Input:             SIMULATED mRNA counts per cell (gamma–Poisson, μ = 6, σ² = 30); typed-in record of 100 coin flips;
                   SIMULATED log fold changes, genotypes, polygenic trait, replicate design, cholesterol trial, images
Output:            running means, sampling distributions, CLT histograms, z / t / bootstrap intervals, sample sizes
Method:            Var(X̄) = σ²/n; Chebyshev; standardisation Z = (X̄ − μ)/(σ/√n); CI = estimate ± multiplier × SE;
                   simulation with a known truth; bootstrap resampling
Key equation:      X̄_n ≈ N(μ, σ²/n);   CI: x̄ ± z σ/√n  or  x̄ ± t_{n−1} s/√n;   n = (zσ/E)²
Key parameters:    sample size n; σ (or s); confidence level; number of bootstrap resamples B
What changed when I modified …  (e.g. n = 5 → 100; Bernoulli p = 0.1 → 0.5; one major-effect SNP; 1.96 → t_{n−1})
One failure mode:  dependent or Cauchy-like data — averaging does not reduce the error as σ/√n promises
Connection to 02-680: the sample mean as a random variable, LLN (ε-form), CLT (variance σ²/n), why use the CLT,
                   statistics in a nutshell, what a statistic is, confidence intervals
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — Ch. 5 (convergence, the law
  of large numbers, the central limit theorem), Ch. 6 (statistical inference, confidence intervals), Ch. 8 (the bootstrap).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — §6.2 (the law of large numbers,
  Markov and Chebyshev inequalities), §6.3 (the central limit theorem), §8.5 (confidence intervals).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — §6.4
  (summary statistics, sums of random variables).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press — the chapters on sampling and
  estimation.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Raj A., Peskin C. S., Tranchina D., Vargas D. Y., Tyagi S. (2006) Stochastic mRNA synthesis in mammalian cells.
  *PLoS Biology* 4:e309 (transcriptional bursting seen by single-molecule FISH).
- Fisher R. A. (1918) The correlation between relatives on the supposition of Mendelian inheritance. *Transactions of the
  Royal Society of Edinburgh* 52:399–433.
- Efron B. (1979) Bootstrap methods: another look at the jackknife. *Annals of Statistics* 7:1–26.
- Felsenstein J. (1985) Confidence limits on phylogenies: an approach using the bootstrap. *Evolution* 39:783–791.
- Blainey P., Krzywinski M., Altman N. (2014) Points of significance: Replication. *Nature Methods* 11:879–880.

## ➡️ Next

**[Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb)** — so far we estimated a mean by averaging and asked how precise the average is. Module 23 asks
the general question: given data and a *model* (Bernoulli, normal, Poisson …) with unknown parameters, which parameter
values explain the data best? The answer — **maximum likelihood** — recovers the sample mean and the allele-counting
estimate of this module as special cases, and its precision is described by the same $1/\sqrt n$ law and normal
approximation that you met here.